<a href="https://colab.research.google.com/github/jasonmcgovern/gallery/blob/main/socsDeathStar_updatedv3_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# UPDATED VERSION - January 2026
# ==============================================================================
# Updates:
# 1. ZIP file filtering - Automatically ignores .zip files during crawl
# 2. Email reporting - Option to email completed reports to specified address
# 3. H1/H2 keyword analysis - Added to AI Content scoring (now 6 factors)
# 4. Indexability analysis - Confirmed in AI search scoring methodology
# ==============================================================================

#@title Cell 1 - Download necessary libraries

%%capture
!pip install selenium beautifulsoup4 requests pandas openpyxl lxml webdriver-manager

In [ ]:
#@title Cell 2 - Initialize script

# Complete Enhanced SEO + AI Search Audit Tool with Distinct Scoring
# Integrated with missing factors from Aleyda Solis's AI Search Optimization Checklist
# NOW WITH PROPER DISTINCT AI CONTENT vs TECHNICAL SCORING

%%capture

import time
import json
import pandas as pd
import numpy as np
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re
from datetime import datetime, timedelta
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders
import os
import warnings
warnings.filterwarnings('ignore')



In [ ]:
#@title Cell 3 - Configure the Audit

# =============================================================================
# CONFIGURATION SECTION - Use the form on the right to input values →
# =============================================================================

#@markdown ### Website Configuration
AnalyzedDomain = "https://www.example.com" #@param {type:"string"}
OutputFilename = "seoAudit.xlsx" #@param {type:"string"}
EmailAddress = "" #@param {type:"string"}

#@markdown ### Sitemap (Optional)
XMLSitemap = "https://www.example.com/sitemap.xml" #@param {type:"string"}

#@markdown ### Keywords for Relevance Analysis
BrandKeywords = "yourbrand" #@param {type:"string"}
UnbrandedKeywords = "keyword one, keyword two, keyword three" #@param {type:"string"}

#@markdown ### Crawl Settings
MaxPages = 500 #@param {type:"slider", min:10, max:5000, step:10}
CrawlDelay = 1.5 #@param {type:"slider", min:0.5, max:5.0, step:0.5}

#@markdown ### PageSpeed Insights API (Optional)
#@markdown Get a free API key from: https://developers.google.com/speed/docs/insights/v5/get-started
PageSpeedAPIKey = "" #@param {type:"string"}
PageSpeedSampleSize = 50 #@param {type:"slider", min:0, max:50, step:5}

#@markdown **Note:** Leave PageSpeedAPIKey empty to use placeholder performance values (faster crawl)

#@markdown ### Title & Meta Description Length Compliance
#@markdown - **Strict**: Title ≤ 60 chars, Meta ≤ 155 chars
#@markdown - **Loose**: Title ≤ 75 chars, Meta ≤ 170 chars
#@markdown - **Ignore**: Length checks disabled entirely
LengthCompliance = "Loose" #@param ["Strict", "Loose", "Ignore"]

# =============================================================================
# Parse configuration
# =============================================================================

domain = AnalyzedDomain
# Normalize filename — ensure .xlsx extension regardless of what was entered
_base = OutputFilename.strip()
if _base.lower().endswith(".xls"):
    _base = _base[:-4]  # strip legacy .xls
elif _base.lower().endswith(".xlsx"):
    _base = _base[:-5]  # strip so we re-add cleanly
nameOfNewSpreadSheet = _base + ".xlsx"
email_address = EmailAddress.strip() if EmailAddress.strip() else None
branKeywords = [keyword.strip() for keyword in BrandKeywords.split(",")]
unbKeywords = [keyword.strip() for keyword in UnbrandedKeywords.split(",")]
siteMapAddress = XMLSitemap

# PageSpeed API configuration
pagespeed_api_key = PageSpeedAPIKey if PageSpeedAPIKey.strip() else None
pagespeed_sample_size = PageSpeedSampleSize if pagespeed_api_key else 0

# Length compliance thresholds
_length_thresholds = {
    "Strict": {"title_min": 30, "title_max": 60, "meta_min": 120, "meta_max": 155},
    "Loose":  {"title_min": 30, "title_max": 75, "meta_min": 120, "meta_max": 170},
    "Ignore": None,
}
length_thresholds = _length_thresholds.get(LengthCompliance, _length_thresholds["Strict"])
print(f"Length Compliance: {LengthCompliance}")
if length_thresholds:
    print(f"  Title:  {length_thresholds['title_min']}–{length_thresholds['title_max']} chars")
    print(f"  Meta:   {length_thresholds['meta_min']}–{length_thresholds['meta_max']} chars")
else:
    print("  Length checks disabled")

print("="*70)
print("AUDIT CONFIGURATION LOADED")
print("="*70)
print(f"Domain: {domain}")
print(f"Output File: {nameOfNewSpreadSheet} (resolved from: '{OutputFilename}')")
print(f"Max Pages: {MaxPages}")
print(f"Crawl Delay: {CrawlDelay}s")
print(f"\nBrand Keywords: {branKeywords}")
print(f"Unbranded Keywords: {unbKeywords[:3]}{'...' if len(unbKeywords) > 3 else ''}")

if pagespeed_api_key:
    print(f"\n✅ PageSpeed Insights API: ENABLED")
    print(f"   Will collect real performance data for {pagespeed_sample_size} pages")
    print(f"   Note: This will slow down the crawl (~3-5 seconds per sampled page)")
else:
    print(f"\n⚠️ PageSpeed Insights API: DISABLED")
    print(f"   Using placeholder performance values (faster crawl)")
    print(f"   Get a free API key: https://developers.google.com/speed/docs/insights/v5/get-started")

print("="*70)

In [ ]:
#@title Cell 4 - Set up functions

# =============================================================================
# ENHANCED AI BOT CRAWLABILITY ANALYSIS
# =============================================================================
def send_email_report(recipient_email, excel_filename, domain, ai_scores=None, crawl_stats=None):
    """Send the completed audit report via email with performance summary"""
    try:
        # Gmail SMTP configuration
        smtp_server = "smtp.gmail.com"
        smtp_port = 587
        sender_email = "YOUR_GMAIL@gmail.com"  # Update with your Gmail
        sender_password = "YOUR_GMAIL_APP_PASSWORD"   # Update with your Gmail App Password

        # Create message
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = recipient_email
        msg['Subject'] = f'SEO & AI Search Audit Complete - {domain}'

        # Build email body with summary
        body = f"""
SEO & AI SEARCH AUDIT REPORT
{'='*60}

Domain: {domain}
Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

{'='*60}
AUDIT SUMMARY
{'='*60}
"""

        # Add crawl statistics if provided
        if crawl_stats:
            body += f"""
CRAWL STATISTICS:
  • Total Pages Analyzed: {crawl_stats.get('total_pages', 'N/A')}
  • Indexable Pages: {crawl_stats.get('indexable_pages', 'N/A')}
  • Average Links per Page: {crawl_stats.get('avg_links', 'N/A')}

"""

        # Add AI scores if provided
        if ai_scores:
            body += f"""
TRADITIONAL SEO PERFORMANCE:
  • SEO Score: {ai_scores.get('traditional_seo_score', 'N/A'):.1%}
  • Total SEO Errors: {ai_scores.get('total_seo_errors', 'N/A')}
  • Errors per Page: {ai_scores.get('errors_per_page', 'N/A'):.1f}

AI SEARCH READINESS SCORES:
  • AI Content Score: {ai_scores.get('ai_content_score', 0):.1%}
  • AI Technical Score: {ai_scores.get('ai_technical_score', 0):.1%}
  • AI Combined Score: {ai_scores.get('ai_combined_score', 0):.1%}

AI ISSUE BREAKDOWN:
  • Content Issues: {ai_scores.get('content_issues', 0)} / {ai_scores.get('total_pages', 0) * 6} possible
  • Technical Issues: {ai_scores.get('technical_issues', 0)} / {ai_scores.get('total_pages', 0) * 4} possible
  • Total AI Issues: {ai_scores.get('total_ai_issues', 0)} / {ai_scores.get('total_pages', 0) * 10} possible

AI OPTIMIZATION GRADES:
  • Content Grade: {ai_scores.get('content_grade', 'N/A')}
  • Technical Grade: {ai_scores.get('technical_grade', 'N/A')}
  • Overall Grade: {ai_scores.get('combined_grade', 'N/A')}

"""

        body += f"""
{'='*60}
DETAILED ANALYSIS
{'='*60}

The comprehensive audit report is attached as an Excel file.

This audit includes:
  ✅ Traditional SEO optimization (21 factors)
  ✅ AI Content optimization scoring (6 factors)
  ✅ AI Technical optimization scoring (4 factors)
  ✅ Combined AI readiness scoring (10 factors)
  ✅ Strategic recommendations
  ✅ Page-by-page analysis

{'='*60}

Best regards,
SEO Audit System
"""

        msg.attach(MIMEText(body, 'plain'))

        # Attach Excel file
        if os.path.exists(excel_filename):
            with open(excel_filename, 'rb') as attachment:
                part = MIMEBase('application', 'octet-stream')
                part.set_payload(attachment.read())
                encoders.encode_base64(part)
                part.add_header(
                    'Content-Disposition',
                    f'attachment; filename= {os.path.basename(excel_filename)}'
                )
                msg.attach(part)

        # Send email
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(sender_email, sender_password)
        server.send_message(msg)
        server.quit()

        print(f"\n📧 Email sent successfully to {recipient_email}")
        print(f"   ✓ Performance summary included in email body")
        print(f"   ✓ Excel report attached")
        return True

    except Exception as e:
        print(f"\n⚠️ Failed to send email: {str(e)}")
        print(f"   Note: Update sender credentials in send_email_report() function")
        return False

def analyze_ai_bot_crawlability(domain):
    """Analyze robots.txt for AI bot accessibility"""
    try:
        robots_url = urljoin(domain, '/robots.txt')
        response = requests.get(robots_url, timeout=10)
        robots_content = response.text.lower()

        # AI bots to check for
        ai_bots = {
            'gptbot': 'OpenAI GPT crawler',
            'google-extended': 'Google AI training',
            'bingbot': 'Microsoft/Bing AI',
            'claudebot': 'Anthropic Claude',
            'claude-web': 'Claude web crawler',
            'ccbot': 'Common Crawl (used by AI)',
            'perplexitybot': 'Perplexity AI'
        }

        results = {
            'robots_txt_exists': response.status_code == 200,
            'ai_bots_blocked': [],
            'ai_bots_allowed': [],
            'total_ai_bots_blocked': 0
        }

        if response.status_code == 200:
            for bot, description in ai_bots.items():
                if f'user-agent: {bot}' in robots_content:
                    if 'disallow: /' in robots_content:
                        results['ai_bots_blocked'].append(bot)
                        results['total_ai_bots_blocked'] += 1
                    else:
                        results['ai_bots_allowed'].append(bot)

        return results

    except Exception as e:
        return {
            'robots_txt_exists': False,
            'ai_bots_blocked': [],
            'ai_bots_allowed': [],
            'total_ai_bots_blocked': 0,
            'error': str(e)
        }



In [ ]:
#@title Cell 5 - Simplified Crawler (Requests-Based - No Chrome Required)

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
from collections import deque
import time
import pandas as pd
from datetime import datetime
import re
import json
import warnings
warnings.filterwarnings('ignore')

class SimplifiedCrawler:
    """Simplified crawler using requests instead of Selenium - More accurate for AI crawler simulation"""

    def __init__(self, domain, max_pages=50, crawl_delay=1.0, pagespeed_api_key=None, sample_size=10):
        self.domain = domain
        self.max_pages = max_pages
        self.crawl_delay = crawl_delay
        self.pagespeed_api_key = pagespeed_api_key
        self.sample_size = sample_size  # Number of pages to get real PageSpeed data for

        # Crawling state
        self.crawl_data = []
        self.visited_urls = set()
        self.crawl_queue = deque()
        self.failed_urls = set()
        # Maps each target URL to the set of source URLs that link to it.
        # Built during crawl, inverted post-crawl to produce true inlink counts.
        self.outbound_link_map = {}  # {target_url: set(source_urls)}

        # Domain validation
        self.domain_netloc = urlparse(domain).netloc

        # Session for persistent connections
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': '*/*',  # More permissive - accept anything
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Cache-Control': 'max-age=0',
            'Sec-Ch-Ua': '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
            'Sec-Ch-Ua-Mobile': '?0',
            'Sec-Ch-Ua-Platform': '"Windows"',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Sec-Fetch-User': '?1',
            'Upgrade-Insecure-Requests': '1',
            'DNT': '1'
        })

        # Robots.txt handling
        self.robots_rules = self.fetch_and_parse_robots_txt()

        # URL patterns to avoid
        self.avoid_patterns = [
            r'\\.pdf$', r'\\.jpg$', r'\\.jpeg$', r'\\.png$', r'\\.gif$', r'\\.svg$',
            r'\\.css$', r'\\.js$', r'\\.xml$', r'\\.json$', r'\\.zip$', r'\\.doc$',
            r'\\.docx$', r'\\.xls$', r'\\.xlsx$', r'\\.ppt$', r'\\.pptx$',
            r'mailto:', r'tel:', r'javascript:', r'#', r'\\?print=', r'/print/',
            r'\\.rss$', r'\\.atom$', r'/feed/', r'/rss/', r'/sitemap'
        ]

        print(f"✅ Simplified crawler initialized for: {self.domain_netloc}")
        print(f"   Max pages: {self.max_pages}, Crawl delay: {self.crawl_delay}s")
        if self.pagespeed_api_key:
            print(f"   PageSpeed API enabled for {self.sample_size} sample pages")
        else:
            print(f"   PageSpeed API disabled (using placeholder values)")

    def fetch_and_parse_robots_txt(self):
        """Fetch and parse robots.txt for User-agent: *"""
        robots_url = urljoin(self.domain, '/robots.txt')
        rules = {'disallow': [], 'allow': []}

        try:
            response = self.session.get(robots_url, timeout=10)
            if response.status_code == 200:
                print(f"✅ Robots.txt loaded from {robots_url}")
                lines = response.text.splitlines()
                parse_global = False

                for line in lines:
                    line = line.strip()
                    if not line or line.startswith('#'):
                        continue

                    if line.lower().startswith('user-agent:'):
                        agent = line[len('user-agent:'):].strip().lower()
                        parse_global = (agent == '*')
                    elif parse_global:
                        if line.lower().startswith('disallow:'):
                            path = line[len('disallow:'):].strip()
                            if path:
                                rules['disallow'].append(path)
                        elif line.lower().startswith('allow:'):
                            path = line[len('allow:'):].strip()
                            if path:
                                rules['allow'].append(path)
        except Exception as e:
            print(f"⚠️ Could not fetch robots.txt: {e}")

        return rules

    def can_fetch(self, url):
        """Check if URL is allowed by robots.txt"""
        parsed_url = urlparse(url)
        path = parsed_url.path
        if parsed_url.query:
            path += '?' + parsed_url.query

        if not path.startswith('/'):
            path = '/' + path

        # Check Allow rules first (more specific)
        for allowed_path in sorted(self.robots_rules.get('allow', []), key=len, reverse=True):
            if path.startswith(allowed_path):
                if allowed_path == '/' and path != '/':
                    continue
                return True

        # Check Disallow rules
        for disallowed_path in sorted(self.robots_rules.get('disallow', []), key=len, reverse=True):
            if path.startswith(disallowed_path):
                return False

        return True

    def is_crawlable_url(self, url):
        """Check if URL is crawlable"""
        parsed_url = urlparse(url)

        # Check domain
        if parsed_url.netloc != self.domain_netloc:
            return False

        # Check robots.txt
        if not self.can_fetch(url):
            return False

        # Check avoid patterns
        for pattern in self.avoid_patterns:
            if re.search(pattern, url, re.IGNORECASE):
                return False

        return True

    def normalize_url(self, url):
        """Normalize URL for deduplication.
        Preserves trailing slashes exactly as linked — stripping them caused
        false Non-Indexable flags when the stored Address (no slash) was
        compared against a canonical tag (with slash) for the same page.
        Deduplication of slash vs no-slash variants is handled by checking
        visited_urls against both forms before queuing.
        """
        parsed = urlparse(url)
        normalized = urlunparse((
            parsed.scheme, parsed.netloc, parsed.path,
            parsed.params, parsed.query, None
        ))
        return normalized

    def get_pagespeed_data(self, url):
        """Get real Core Web Vitals from PageSpeed Insights API"""
        if not self.pagespeed_api_key:
            # Return placeholder values
            return {
                'Performance Score': 50,
                'CrUX Largest Contentful Paint Time (ms)': 2500,
                'CrUX Cumulative Layout Shift': 0.1,
                'CrUX Interaction to Next Paint (ms)': 200
            }

        try:
            api_url = f"https://www.googleapis.com/pagespeedonline/v5/runPagespeed"
            params = {
                'url': url,
                'key': self.pagespeed_api_key,
                'category': 'performance'
            }

            response = requests.get(api_url, params=params, timeout=30)

            if response.status_code == 200:
                data = response.json()

                # Extract performance score
                perf_score = data.get('lighthouseResult', {}).get('categories', {}).get('performance', {}).get('score', 0.5)

                # Extract CrUX metrics (real user data)
                crux_data = data.get('loadingExperience', {}).get('metrics', {})

                lcp = crux_data.get('LARGEST_CONTENTFUL_PAINT_MS', {}).get('percentile', 2500)
                cls = crux_data.get('CUMULATIVE_LAYOUT_SHIFT_SCORE', {}).get('percentile', 0.1) / 100
                inp = crux_data.get('INTERACTION_TO_NEXT_PAINT', {}).get('percentile', 200)

                return {
                    'Performance Score': int(perf_score * 100),
                    'CrUX Largest Contentful Paint Time (ms)': lcp,
                    'CrUX Cumulative Layout Shift': cls,
                    'CrUX Interaction to Next Paint (ms)': inp
                }
            else:
                # API error, return placeholders
                return {
                    'Performance Score': 50,
                    'CrUX Largest Contentful Paint Time (ms)': 2500,
                    'CrUX Cumulative Layout Shift': 0.1,
                    'CrUX Interaction to Next Paint (ms)': 200
                }
        except Exception as e:
            print(f"  ⚠️ PageSpeed API error: {str(e)[:50]}")
            return {
                'Performance Score': 50,
                'CrUX Largest Contentful Paint Time (ms)': 2500,
                'CrUX Cumulative Layout Shift': 0.1,
                'CrUX Interaction to Next Paint (ms)': 200
            }

    def extract_page_data(self, url, get_real_performance=False, crawl_depth=0):
        """Extract comprehensive page data using requests.

        Uses allow_redirects=False on the initial request to capture the
        true HTTP status code. If a redirect is detected, the Location
        header is recorded and a second request follows the chain to fetch
        content for SEO extraction. The original URL is always stored as
        Address so redirect rows are correctly identified in the output.
        """
        try:
            # First request — do NOT follow redirects so we capture true status
            initial = self.session.get(url, timeout=15, allow_redirects=False)
            true_status = initial.status_code
            redirect_url = ''

            if true_status in (301, 302, 307, 308):
                # Record redirect destination from Location header
                redirect_url = initial.headers.get('Location', '')
                # Follow the full chain for content extraction only
                try:
                    response = self.session.get(url, timeout=15, allow_redirects=True)
                except Exception:
                    response = initial  # fallback to initial if follow fails
            else:
                response = initial

            # Parse whatever HTML we received regardless of status code.
            content_type = response.headers.get('content-type', 'text/html')
            try:
                soup = BeautifulSoup(response.content, 'html.parser')
            except Exception:
                soup = BeautifulSoup('', 'html.parser')

            # Basic page data — always 'OK' so page enters crawl_data.
            # true_status reflects the actual response before any redirect.
            page_data = {
                'Address': url,
                'Content Type': content_type,
                'Status Code': true_status,
                'Redirect URL': redirect_url,
                'Status': 'OK',
                'Crawl Timestamp': datetime.now().isoformat()
            }

            # Extract SEO data
            page_data.update(self.extract_traditional_seo_data(soup, url))
            page_data.update(self.extract_ai_search_data(soup))
            page_data.update(self.extract_citation_worthiness(soup, url))
            page_data.update(self.extract_multimodal_content(soup))
            page_data.update(self.extract_freshness_signals(soup))
            page_data.update(self.extract_chunk_optimization(soup))
            page_data.update(self.extract_enhanced_technical_factors(soup, url, response.text))

            # Performance data (real or placeholder)
            if get_real_performance:
                print(f"  🔍 Getting real PageSpeed data for {url[:50]}...")
                page_data.update(self.get_pagespeed_data(url))
            else:
                page_data.update({
                    'Performance Score': 50,
                    'CrUX Largest Contentful Paint Time (ms)': 2500,
                    'CrUX Cumulative Layout Shift': 0.1,
                    'CrUX Interaction to Next Paint (ms)': 200
                })

            # Indexability
            x_robots = response.headers.get('X-Robots-Tag', '') if hasattr(response, 'headers') else ''
            page_data['Address'] = url  # ensure Address is set before indexability check
            page_data['Indexability'] = self.assess_indexability(page_data, x_robots)
            page_data['Indexability Status'] = page_data['Indexability']

            # Discover links
            links_discovered = self.discover_links(url, soup, crawl_depth)
            page_data['Links Discovered'] = links_discovered

            return page_data

        except Exception as e:
            return self.create_error_record(url, str(e), 500)

    def discover_links(self, current_url, soup, current_depth=0):
        """Discover and queue new links from current page.
        Queues each new URL as a (url, depth+1) tuple for BFS depth tracking.
        """
        discovered = set()
        links = soup.find_all('a', href=True)

        for link in links:
            href = link.get('href', '').strip()
            if not href:
                continue

            absolute_url = urljoin(current_url, href)
            normalized_url = self.normalize_url(absolute_url)

            # Exact URL deduplication — trailing slash variants are treated
            # as separate pages and will each appear as their own row.
            if (self.is_crawlable_url(normalized_url) and
                normalized_url not in self.visited_urls and
                normalized_url not in self.failed_urls and
                normalized_url not in self.crawl_queue):
                discovered.add(normalized_url)
                self.crawl_queue.append((normalized_url, current_depth + 1))

            # Record this outbound link regardless of whether it's new,
            # so we can invert the map post-crawl for true inlink counts.
            if self.is_same_domain(normalized_url):
                if normalized_url not in self.outbound_link_map:
                    self.outbound_link_map[normalized_url] = set()
                self.outbound_link_map[normalized_url].add(current_url)

        return len(discovered)

    def traditional_crawl(self, start_url, progress_callback=None):
        """Perform traditional web crawling"""
        start_url = self.normalize_url(start_url)
        self.crawl_queue.append((start_url, 0))  # (url, crawl_depth)

        print(f"\nStarting traditional crawl from: {start_url}")
        print(f"Target: {self.max_pages} pages\n")

        crawl_count = 0
        pagespeed_count = 0

        while self.crawl_queue and crawl_count < self.max_pages:
            current_url, current_depth = self.crawl_queue.popleft()

            if current_url in self.visited_urls:
                continue

            self.visited_urls.add(current_url)
            crawl_count += 1

            if progress_callback:
                progress_callback(crawl_count, self.max_pages, current_url)

            if crawl_count > 1:
                time.sleep(self.crawl_delay)

            # Determine if we should get real PageSpeed data for this page
            get_real_performance = (
                self.pagespeed_api_key and
                pagespeed_count < self.sample_size
            )

            if get_real_performance:
                pagespeed_count += 1

            # Extract page data
            page_data = self.extract_page_data(current_url, get_real_performance, current_depth)

            # All pages enter crawl_data regardless of status code so that
            # non-200 responses appear in the Excel output.
            page_data['Crawl Depth'] = current_depth
            self.crawl_data.append(page_data)
            status_code = page_data.get('Status Code', 0)
            if page_data.get('Error'):
                self.failed_urls.add(current_url)
                print(f"❌ [{crawl_count:3d}] {current_url[:70]}... (ERROR: {page_data.get('Error', 'Unknown')})")
            elif status_code == 200:
                print(f"✅ [{crawl_count:3d}] {current_url[:70]}... (+{page_data.get('Links Discovered', 0)} links)")
            else:
                print(f"⚠️  [{crawl_count:3d}] {current_url[:70]}... (HTTP {status_code})")

        print(f"\nCrawl completed!")
        print(f"Pages crawled: {len(self.crawl_data)}")
        print(f"Pages failed: {len(self.failed_urls)}")
        print(f"Queue remaining: {len(self.crawl_queue)}")
        if self.pagespeed_api_key:
            print(f"Real PageSpeed data collected for: {pagespeed_count} pages")

        # ---------------------------------------------------------------
        # POST-CRAWL: invert outbound_link_map to produce true inlink counts
        # ---------------------------------------------------------------
        print(f"\nCalculating true inlink counts from {len(self.outbound_link_map)} linked URLs...")
        for page in self.crawl_data:
            url = page.get('Address', '')
            # Normalize with and without trailing slash so both variants match
            inlinkers = (
                self.outbound_link_map.get(url, set()) |
                self.outbound_link_map.get(url.rstrip('/'), set()) |
                self.outbound_link_map.get(url.rstrip('/') + '/', set())
            )
            page['Unique Inlinks'] = len(inlinkers)

        crawl_df = pd.DataFrame(self.crawl_data)
        crawl_df = self.ensure_compatible_columns(crawl_df)

        return crawl_df

    def is_same_domain(self, url):
        """Check if a URL belongs to the crawl domain."""
        try:
            return urlparse(url).netloc == self.domain_netloc
        except:
            return False

    def extract_traditional_seo_data(self, soup, url):
        """Extract SEO data in exact format expected by existing functions"""
        data = {}

        # Title tag analysis
        title_tag = soup.find('title')
        title_text = title_tag.text.strip() if title_tag else ""
        data['Title 1'] = title_text
        data['Title 1 Length'] = len(title_text)
        data['Title 1 Pixel Width'] = len(title_text) * 6

        # Meta description
        meta_desc = soup.find('meta', attrs={'name': 'description'})
        desc_text = meta_desc.get('content', '').strip() if meta_desc else ""
        data['Meta Description 1'] = desc_text
        data['Meta Description 1 Length'] = len(desc_text)
        data['Meta Description 1 Pixel Width'] = len(desc_text) * 6

        # Canonical URL
        canonicals = soup.find_all('link', attrs={'rel': 'canonical'})
        data['Canonical Link Element 1'] = canonicals[0].get('href', '') if canonicals else ""
        data['Canonical Count'] = len(canonicals)

        # Robots meta tag
        robots_meta = soup.find('meta', attrs={'name': 'robots'})
        data['Meta Robots 1'] = robots_meta.get('content', '') if robots_meta else ""

        # Headings analysis
        data.update(self.extract_headings_compatible(soup))

        # Content metrics
        data.update(self.extract_content_metrics_compatible(soup))

        # Links analysis
        data.update(self.extract_links_compatible(soup, url))

        # Images analysis
        data.update(self.extract_images_compatible(soup))

        # URL analysis
        data['Crawl Depth'] = self.calculate_crawl_depth(url)

        return data

    def extract_headings_compatible(self, soup):
        """Extract headings in exact format expected by existing functions"""
        data = {}

        for level in range(1, 7):
            headings = soup.find_all(f'h{level}')
            for i in range(1, 3):
                key = f'H{level}-{i}'
                if i <= len(headings):
                    data[key] = headings[i-1].get_text().strip()
                else:
                    data[key] = ""

        return data

    def extract_content_metrics_compatible(self, soup):
        """Extract content metrics"""
        # Get text content
        text = soup.get_text(separator=' ', strip=True)
        words = text.split()

        return {
            'Word Count': len(words),
            'Text Ratio': len(text) / max(len(str(soup)), 1)
        }

    def extract_links_compatible(self, soup, url):
        """Extract link metrics"""
        all_links = soup.find_all('a', href=True)

        internal_links = []
        external_links = []
        current_domain = urlparse(url).netloc

        for link in all_links:
            href = link.get('href', '')
            if href.startswith('http'):
                link_domain = urlparse(href).netloc
                if link_domain == current_domain:
                    internal_links.append(href)
                else:
                    external_links.append(href)
            elif not href.startswith(('mailto:', 'tel:', 'javascript:', '#')):
                internal_links.append(href)

        return {
            'Unique Outlinks (Internal)': len(set(internal_links)),
            'Unique Outlinks (External)': len(set(external_links)),
            'Total Links': len(all_links)
        }

    def extract_images_compatible(self, soup):
        """Extract image metrics"""
        images = soup.find_all('img')
        images_with_alt = sum(1 for img in images if img.get('alt'))

        return {
            'Images': len(images),
            'Images Missing Alt Text': len(images) - images_with_alt
        }

    def extract_ai_search_data(self, soup):
        """Extract AI search optimization data"""
        data = {}

        # Structured data extraction
        json_ld_scripts = soup.find_all('script', type='application/ld+json')
        schema_types = set()

        for script in json_ld_scripts:
            try:
                structured = json.loads(script.string)
                if isinstance(structured, dict) and '@type' in structured:
                    schema_types.add(structured['@type'])
                elif isinstance(structured, list):
                    for item in structured:
                        if isinstance(item, dict) and '@type' in item:
                            schema_types.add(item['@type'])
            except:
                continue

        # Microdata extraction
        microdata_items = soup.find_all(attrs={'itemtype': True})
        for item in microdata_items:
            itemtype = item.get('itemtype', '')
            if 'schema.org' in itemtype:
                schema_type = itemtype.split('/')[-1]
                schema_types.add(schema_type)

        data['AI_Schema_Types'] = ','.join(schema_types)
        data['AI_Has_Structured_Data'] = len(schema_types) > 0
        data['AI_Schema_Count'] = len(schema_types)

        # Content structure for AI
        semantic_elements = ['article', 'section', 'nav', 'aside', 'header', 'footer', 'main']
        semantic_count = sum(len(soup.find_all(tag)) for tag in semantic_elements)
        data['AI_Semantic_Elements'] = semantic_count

        # Lists and structured content
        lists = soup.find_all(['ul', 'ol', 'dl'])
        data['AI_List_Elements'] = len(lists)

        # FAQ/Answer content detection
        text_content = soup.get_text().lower()
        question_patterns = ['what is', 'how to', 'why does', 'when should', 'where can']
        question_count = sum(1 for pattern in question_patterns if pattern in text_content)
        data['AI_Question_Content'] = question_count

        return data

    def extract_citation_worthiness(self, soup, url):
        """Extract citation-worthiness factors"""
        # Author detection
        author_schema = soup.find('script', type='application/ld+json')
        has_author_schema = False

        if author_schema:
            try:
                schema_data = json.loads(author_schema.string)
                if 'author' in str(schema_data).lower():
                    has_author_schema = True
            except:
                pass

        # Author meta tags and bylines
        author_meta = soup.find('meta', attrs={'name': re.compile(r'author', re.I)})
        has_author_meta = bool(author_meta)

        byline_patterns = [r'by\\s+[A-Z][a-z]+\\s+[A-Z][a-z]+', r'author:\\s*[A-Z][a-z]+', r'written by\\s+[A-Z][a-z]+']
        text_content = soup.get_text()
        has_byline = any(re.search(pattern, text_content, re.I) for pattern in byline_patterns)

        # External citations
        domain_current = urlparse(url).netloc
        external_links = soup.find_all('a', href=True)

        authoritative_domains = [
            'ncbi.nlm.nih.gov', 'pubmed.ncbi.nlm.nih.gov', 'scholar.google.com',
            'doi.org', 'arxiv.org', 'jstor.org', 'researchgate.net',
            'who.int', 'cdc.gov', 'fda.gov', 'nih.gov', 'gov.uk'
        ]

        external_citation_count = 0
        authoritative_citation_count = 0

        for link in external_links:
            href = link.get('href', '')
            if href.startswith('http'):
                link_domain = urlparse(href).netloc
                if link_domain != domain_current:
                    external_citation_count += 1
                    if any(auth_domain in link_domain for auth_domain in authoritative_domains):
                        authoritative_citation_count += 1

        # Specific claims detection
        specific_claims = len(re.findall(r'\\d+%|\\d+\\.\\d+%|\\$\\d+|\\d+,\\d+|\\d+ million|\\d+ billion', text_content.lower()))

        return {
            'AI_Has_Author_Schema': has_author_schema,
            'AI_Has_Author_Meta': has_author_meta,
            'AI_Has_Byline': has_byline,
            'AI_External_Citations': external_citation_count,
            'AI_Authoritative_Citations': authoritative_citation_count,
            'AI_Specific_Claims': specific_claims,
            'AI_Author_Signals': sum([has_author_schema, has_author_meta, has_byline])
        }

    def extract_multimodal_content(self, soup):
        """Extract multi-modal content analysis"""
        # Video analysis
        videos = soup.find_all('video')
        video_with_captions = sum(1 for video in videos if video.find('track') or video.get('captions'))

        # Table analysis
        tables = soup.find_all('table')
        proper_table_markup = sum(1 for table in tables if table.find('th') or table.find('caption'))

        # Figure/caption analysis
        figures = soup.find_all('figure')
        figures_with_captions = sum(1 for fig in figures if fig.find('figcaption'))

        # Images of tables detection (problematic)
        images = soup.find_all('img')
        table_images = sum(1 for img in images
                          if any(word in img.get('alt', '').lower() or word in img.get('src', '').lower()
                                for word in ['table', 'chart', 'graph', 'data']))

        return {
            'AI_Video_Count': len(videos),
            'AI_Videos_With_Captions': video_with_captions,
            'AI_Table_Count': len(tables),
            'AI_Proper_Table_Markup': proper_table_markup,
            'AI_Figure_Count': len(figures),
            'AI_Figures_With_Captions': figures_with_captions,
            'AI_Table_Images_Count': table_images
        }

    def extract_freshness_signals(self, soup):
        """Extract content freshness signals"""
        # Check meta tags for dates
        date_meta_tags = soup.find_all('meta', attrs={'name': re.compile(r'date|time|publish|modif', re.I)})
        has_meta_dates = len(date_meta_tags) > 0

        # Check for schema.org datePublished/dateModified
        has_schema_dates = False
        schema_script = soup.find('script', type='application/ld+json')
        if schema_script:
            try:
                schema_data = json.loads(schema_script.string)
                schema_str = str(schema_data).lower()
                if 'datepublished' in schema_str or 'datemodified' in schema_str:
                    has_schema_dates = True
            except:
                pass

        # Look for visible date patterns
        date_patterns = [
            r'published:?\\s*\\d{1,2}[\\/\\-]\\d{1,2}[\\/\\-]\\d{2,4}',
            r'updated:?\\s*\\d{1,2}[\\/\\-]\\d{1,2}[\\/\\-]\\d{2,4}',
            r'last modified:?\\s*\\d{1,2}[\\/\\-]\\d{1,2}[\\/\\-]\\d{2,4}',
            r'\\d{1,2}\\s+(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\\w*\\s+\\d{2,4}'
        ]

        text_content = soup.get_text().lower()
        visible_dates = sum(1 for pattern in date_patterns if re.search(pattern, text_content))

        return {
            'AI_Has_Meta_Dates': has_meta_dates,
            'AI_Has_Schema_Dates': has_schema_dates,
            'AI_Visible_Date_Count': visible_dates,
            'AI_Total_Date_Signals': int(has_meta_dates) + int(has_schema_dates) + visible_dates
        }

    def extract_chunk_optimization(self, soup):
        """Extract chunk-level optimization data"""
        # Section analysis
        sections = soup.find_all(['section', 'article', 'div'], class_=re.compile(r'section|content|post', re.I))
        if not sections:
            sections = soup.find_all(['h2', 'h3'])

        # Subheading coverage analysis
        major_headings = soup.find_all(['h2', 'h3'])
        text_content = soup.get_text()

        if len(text_content.split()) > 0:
            words_per_heading = len(text_content.split()) / max(len(major_headings), 1)
        else:
            words_per_heading = 0

        # Independent section analysis
        sections_with_context = 0
        for section in sections[:10]:
            section_text = section.get_text()
            if len(section_text.split()) > 50 and section.find(['h2', 'h3', 'h4']):
                sections_with_context += 1

        return {
            'AI_Total_Sections': len(sections),
            'AI_Sections_With_Context': sections_with_context,
            'AI_Words_Per_Heading': words_per_heading,
            'AI_Major_Headings': len(major_headings)
        }

    def extract_enhanced_technical_factors(self, soup, url, page_source):
        """Extract additional technical factors"""
        # JavaScript analysis
        script_tags = soup.find_all('script')
        external_scripts = len([s for s in script_tags if s.get('src')])

        total_page_size = len(page_source)
        js_size = sum(len(s.get_text()) for s in script_tags)
        js_percentage = (js_size / total_page_size) * 100 if total_page_size > 0 else 0

        # Check for nosnippet meta robots
        robots_meta = soup.find('meta', attrs={'name': 'robots'})
        has_nosnippet = 'nosnippet' in robots_meta.get('content', '').lower() if robots_meta else False

        return {
            'AI_External_Scripts': external_scripts,
            'AI_JS_Percentage': js_percentage,
            'AI_Has_NoSnippet': has_nosnippet,
            'AI_Total_Elements': len(soup.find_all()),
            'HTML Size (bytes)': total_page_size
        }

    def calculate_crawl_depth(self, url):
        """Calculate crawl depth like Screaming Frog"""
        parsed_url = urlparse(url)
        path = parsed_url.path
        if path == '/' or path == '':
            return 0
        return len([p for p in path.split('/') if p]) - 1

    def assess_indexability(self, data, x_robots_tag=''):
        """Assess indexability against all standard signals.

        Checks in priority order:
        1. Non-200 HTTP status
        2. noindex in meta robots tag
        3. noindex in X-Robots-Tag response header
        4. Canonical pointing to a different URL (page defers indexing to target)
        5. Multiple canonical tags (Google ignores all — effectively no signal)
        """
        # 1. Non-200 status
        status_code = data.get('Status Code', 0)
        if status_code != 200:
            return 'Non-Indexable'

        # 2. Meta robots noindex
        robots_meta = data.get('Meta Robots 1', '').lower()
        if 'noindex' in robots_meta:
            return 'Non-Indexable'

        # 3. X-Robots-Tag response header noindex
        if 'noindex' in str(x_robots_tag).lower():
            return 'Non-Indexable'

        # 4. Canonical pointing to a different URL
        # Exact case-insensitive comparison. normalize_url() no longer strips
        # trailing slashes, so Address and canonical are compared as-crawled.
        # /locations and /locations/ are treated as genuinely different URLs
        # only if the site actually serves them differently.
        canonical = str(data.get('Canonical Link Element 1', '')).strip()
        page_url = str(data.get('Address', '')).strip()
        if canonical and canonical != 'nan' and len(canonical) > 10:
            if canonical.lower() != page_url.lower():
                return 'Non-Indexable'

        # 5. Multiple canonical tags
        canonical_count = data.get('Canonical Count', 1)
        try:
            if int(canonical_count) > 1:
                return 'Non-Indexable'
        except:
            pass

        return 'Indexable'

    def create_error_record(self, url, error, status_code=0):
        """Create a minimal record for URLs that failed at the network level
        (timeout, DNS failure, connection refused) where no HTTP response
        was received at all. These are still included in crawl_data so they
        appear in the Excel output as identifiable failures.
        """
        return {
            'Address': url,
            'Content Type': 'text/html',
            'Status Code': status_code,
            'Status': 'OK',  # 'OK' so the record enters crawl_data
            'Indexability': 'Non-Indexable',
            'Title 1': '',
            'Meta Description 1': '',
            'Canonical Link Element 1': '',
            'H1-1': '', 'H1-2': '', 'H2-1': '',
            'Word Count': 0,
            'Unique Inlinks': 0,
            'Links Discovered': 0,
            'Crawl Depth': 0,
            'Error': error,
            'Performance Score': 0,
            'CrUX Largest Contentful Paint Time (ms)': 0,
            'CrUX Cumulative Layout Shift': 0,
            'CrUX Interaction to Next Paint (ms)': 0,
        }

    def ensure_compatible_columns(self, df):
        """Ensure all columns expected by existing functions are present"""
        required_columns = {
            'Content Type': 'text/html',
            'Title 1 Length': 0,
            'Meta Description 1 Length': 0,
            'Canonical Link Element 1': '',
            'H1-1': '', 'H1-2': '', 'H2-1': '', 'H2-2': '',
            'Word Count': 0,
            'Unique Inlinks': 0,  # overwritten by post-crawl inversion
            'Performance Score': 50,
            'CrUX Largest Contentful Paint Time (ms)': 2500,
            'CrUX Cumulative Layout Shift': 0.1,
            'CrUX Interaction to Next Paint (ms)': 200,
            'Crawl Depth': 0
        }

        for col, default_val in required_columns.items():
            if col not in df.columns:
                df[col] = default_val

        return df

    def get_crawl_stats(self):
        """Get crawling statistics"""
        return {
            'pages_crawled': len(self.crawl_data),
            'pages_failed': len(self.failed_urls),
            'urls_discovered': len(self.visited_urls) + len(self.crawl_queue),
            'queue_remaining': len(self.crawl_queue),
            'crawl_efficiency': len(self.crawl_data) / len(self.visited_urls) if self.visited_urls else 0
        }

print("✅ SimplifiedCrawler class loaded successfully!")
print("   This crawler simulates AI bot behavior using HTTP requests")
print("   Optional: PageSpeed Insights API for real Core Web Vitals")

In [ ]:
#@title Cell 6 - SEO Analysis Functions

# =============================================================================
# ORIGINAL SEO ANALYSIS FUNCTIONS (from your existing tool)
# =============================================================================

def titleEval(idxability, pageTitle):
    """Checks title tag presence.
    Fails if title is missing, empty, or nan.
    Length is evaluated separately by titleLengthEval.
    Non-Indexable pages auto-pass.
    """
    if idxability != "Indexable":
        return 0

    t = str(pageTitle).strip()
    if t == "" or t == "nan" or len(t) < 1:
        return 1

    return 0


def titleLengthEval(idxability, pageTitle, thresholds=None):
    """Checks title tag length against configured compliance thresholds.
    Pass thresholds=None to disable the check (Ignore mode).
    Non-Indexable pages and missing titles auto-pass.
    """
    if thresholds is None:
        return 0  # Ignore mode — length check disabled

    if idxability != "Indexable":
        return 0

    t = str(pageTitle).strip()
    if t == "" or t == "nan":
        return 0  # missing title is titleEval's concern, not length

    if len(t) < thresholds["title_min"] or len(t) > thresholds["title_max"]:
        return 1

    return 0

def indexEval(idxability, response):
    """Original index evaluation function"""
    xRefResult = 0

    if idxability == "Non-Indexable":
        if int(response) == 200:
            xRefResult = 1

    return xRefResult

def mdEval(idxability, metaDesc):
    """Checks meta description presence.
    Fails if meta description is missing, empty, or nan.
    Length is evaluated separately by mdLengthEval.
    Non-Indexable pages auto-pass.
    """
    if idxability != "Indexable":
        return 0

    m = str(metaDesc).strip()
    if m == "" or m == "nan" or len(m) < 1:
        return 1

    return 0


def mdLengthEval(idxability, metaDesc, thresholds=None):
    """Checks meta description length against configured compliance thresholds.
    Pass thresholds=None to disable the check (Ignore mode).
    Non-Indexable pages and missing descriptions auto-pass.
    """
    if thresholds is None:
        return 0  # Ignore mode — length check disabled

    if idxability != "Indexable":
        return 0

    m = str(metaDesc).strip()
    if m == "" or m == "nan":
        return 0  # missing desc is mdEval's concern, not length

    if len(m) < thresholds["meta_min"] or len(m) > thresholds["meta_max"]:
        return 1

    return 0

def kwEval(idxability, pageTitle, keywords):
    """Keyword evaluation function.
    Returns 0 (pass) if any keyword is found in the text.
    Returns 0 for Non-Indexable pages — content checks are not applicable.
    """
    if idxability != "Indexable":
        return 0

    for keyword in keywords:
        if keyword.lower() in str(pageTitle).lower():
            return 0

    return 1

def titleRelevance(idxability, pageTitle, keywords):
    """Original title relevance function"""
    xRefResult = 0

    if idxability == "Indexable":
        xRefResult = 1
        for keyword in keywords:
            if keyword.lower() in pageTitle.lower():
                xRefResult = 0

    return xRefResult

def httpEval(response, inlinks=0):
    """HTTP response evaluation with link equity consideration.
    Returns 1 (issue) if:
    - Status is not 200 or 301
    - Status is 301 BUT page has 10+ inlinks (architectural redirect issue)
    """
    try:
        code = int(response)
        inlink_count = int(inlinks)

        if code == 200:
            return 0

        # 301 is OK unless it carries significant link equity
        if code == 301:
            if inlink_count >= 10:
                return 1  # Heavily linked redirect = architectural problem
            return 0

        # Everything else (302, 404, 500, 503, etc.) is an issue
        return 1

    except:
        return 1  # Unparseable status = flag as issue


def crawlCascadeFail(status_code, inlinks, in_sitemap="no"):
    """Determine if a live-crawled page should auto-fail all SEO tests.

    Returns True if:
    - Status 400+ (404, 500, 503 etc.) — broken page
    - Status 302 or 307 — temporary redirect (link equity not passed)
    - Status 301 with 10+ inlinks — architectural redirect issue
    - In XML sitemap but returning a non-200 status — intent/reality mismatch
    - Status code is unparseable
    """
    try:
        code = int(status_code)
        inlink_count = int(inlinks)
        is_in_sitemap = str(in_sitemap).lower() == "yes"

        if code >= 400:
            return True

        if code in [302, 307]:
            return True

        if code == 301 and inlink_count >= 10:
            return True

        # Sitemap declares intent to index this page, but status is non-200
        if is_in_sitemap and code != 200:
            return True

        return False

    except:
        return True  # Unparseable = cascade fail

def h1KeywordEval(idxability, h1, branKeywords, unbKeywords):
    """Combined H1 keyword relevance check.
    Passes (0) if the H1 contains ANY brand OR unbranded keyword.
    Fails (1) if no configured keyword is found, or H1 is missing/nan.
    Non-indexable pages are skipped (return 0).
    """
    if idxability != "Indexable":
        return 0

    h1_str = str(h1).strip().lower()
    if h1_str == "" or h1_str == "nan":
        return 1

    all_keywords = branKeywords + unbKeywords
    for keyword in all_keywords:
        if keyword.lower() in h1_str:
            return 0

    return 1


def htmlSizeEval(html_size_bytes, indexability):
    """Flags pages whose uncompressed HTML document exceeds 2MB.
    Large HTML documents waste crawl budget, can delay Googlebot's
    parser, and are a strong signal of unminified or bloated markup.
    Note: this measures the HTML document only, not total page weight
    including images, CSS, or JS loaded by the browser.
    Non-indexable pages are skipped.
    """
    if indexability != "Indexable":
        return 0

    try:
        size = int(html_size_bytes)
        if size > 2_000_000:  # 2MB in bytes
            return 1
    except:
        return 0

    return 0


def inSiteMap(address, listofurls):
    """Original sitemap inclusion check function"""
    xRefResult = "no"
    for page in listofurls:
        if page[-1] == "/":
            page = page[:-1]
        if address[-1] == "/":
            address = address[:-1]
        if address == page:
            xRefResult = "yes"

    return xRefResult

def siteMapEval(smStatus, idxStatus):
    """Original sitemap evaluation function"""
    xRefResult = 0

    # Pages in sitemap that are not indexible
    if smStatus == "yes" and idxStatus == "Non-Indexable":
        xRefResult = 1

    # Pages that are indexible but not in sitemap
    if smStatus == "no" and idxStatus == "Indexable":
        xRefResult = 1

    return xRefResult

def canonicalEval(status, canlink, address, response_list, map_list, indexability="Indexable", canonical_count=1):
    """Canonical tag evaluation with sitemap-based target validation.

    Evaluation order:
    1. Non-200 pages auto-fail — a broken page cannot have a valid canonical.
    2. Non-indexable pages auto-fail — canonical is misconfigured by definition.
    3. Multiple canonical tags → flag (1). Google ignores all canonicals when
       more than one is present, making them worse than useless.
    4. Missing canonical (empty, nan, or < 10 chars) → flag (1).
    5. Near-match to self (>= 0.85 similarity but not exact) → flag (1).
       Catches trailing slash, query param, and case differences.
    6. Self-referential canonical (exact match to page URL) → pass (0).
       This is the ideal canonical pattern.
    7. Points to a different URL → evaluate the target:
       - Target found in sitemap → pass (0). Sitemap is used as a proxy
         for a known-good URL, avoiding a second HTTP request.
       - Target not in sitemap → flag (1). Canonical pointing at an
         unknown or potentially non-existent URL is a risk.
    """
    from difflib import SequenceMatcher

    try:
        code = int(status)
    except:
        return 1

    # Non-200 pages auto-fail — a broken page cannot have a valid canonical
    if code != 200:
        return 1

    # Non-indexable pages auto-fail — canonical is misconfigured or missing
    # by definition if the page is not meant to be indexed
    if str(indexability).strip().lower() != "indexable":
        return 1

    # Multiple canonical tags — Google ignores all of them when duplicates exist
    try:
        if int(canonical_count) > 1:
            return 1
    except:
        pass

    # Missing or malformed canonical
    canlink_str = str(canlink).strip()
    if canlink_str == "" or canlink_str == "nan" or len(canlink_str) < 10:
        return 1

    # Normalize both URLs for comparison
    canonical_norm = canlink_str.rstrip("/").lower()
    address_norm = str(address).strip().rstrip("/").lower()

    # Self-referential — ideal case
    if canonical_norm == address_norm:
        return 0

    # Near-match to self — likely a subtle mistake
    similarity = SequenceMatcher(None, canonical_norm, address_norm).ratio()
    if similarity >= 0.85:
        return 1

    # Points to a different URL — validate target against sitemap
    sitemap_normalized = {u.rstrip("/").lower() for u in map_list}
    if canonical_norm in sitemap_normalized:
        return 0  # Target is a known good URL

    # Target not in sitemap — flag as unknown/potentially bad canonical
    return 1

def internalLinkEval(nolinks, indexability, maxpages):
    """Original internal link evaluation function"""
    xRefResult = 0

    if indexability == "Indexable":
        if maxpages < 21:
            if nolinks < (maxpages*0.70):
                xRefResult = 1
        elif maxpages < 100:
            if nolinks < 15:
                xRefResult = 1
        elif maxpages > 99:
            if nolinks < 51:
                xRefResult = 1

    if indexability == "Non-Indexable":
        if nolinks > (maxpages*0.10):
            xRefResult = 1

    return xRefResult

def contentEval(wordCount, indexability):
    """Original content evaluation function"""
    xRefResult = 0

    if indexability == "Indexable":
        if wordCount < 750:
            xRefResult = 1

    return xRefResult

def headingEval(mainHeader, indexability):
    """Checks H1 presence and minimum length only.
    Fails if H1 is missing, empty, 'nan', or fewer than 5 characters.
    H2 is evaluated separately by h2Eval.
    """
    xRefResult = 0

    if indexability == "Indexable":
        h1 = str(mainHeader).strip()
        if h1 == "" or h1 == "nan" or len(h1) < 5:
            xRefResult = 1

    return xRefResult

def addressEval(address, indexability):
    """Original address evaluation function"""
    xRefResult = 0

    if indexability == "Indexable":
        # HTTPS check
        if address[:5] != "https":
            xRefResult = 1

        # Underscore check
        if "_" in address:
            xRefResult = 1

        # Case check
        for letter in address:
            if letter.isupper() == True:
                xRefResult = 1

        # Parameter check
        if address.count('&') > 2:
            xRefResult = 1

        # Length check
        if len(address) > 200:
            xRefResult = 1

        # Session parameter check
        if "session" in address:
            xRefResult = 1

    return xRefResult

def speedEval(lcp, cls, fid, indexability):
    """Original speed evaluation function"""
    xRefResult = 0

    if indexability == "Indexable":
        if lcp > 2500:
            xRefResult = 1
        if cls > 0.1:
            xRefResult = 1
        if fid > 100:
            xRefResult = 1

    return xRefResult

def duplicationEval(item, listOfElements, indexability):
    """Original duplication evaluation function"""
    xRefResult = 0

    if indexability == "Indexable":
        dupeCount = 0
        for element in listOfElements:
            if item == element:
                dupeCount = dupeCount + 1

        if dupeCount > 1:
            xRefResult = 1

    return xRefResult

def multipleH1Eval(idxability, h1_2):
    """Flags pages that have more than one H1 tag.
    H1-2 is populated by the crawler whenever a second H1 exists.
    Multiple H1s create ambiguity about the primary topic of the page
    and are generally considered an SEO anti-pattern.
    Non-Indexable pages auto-pass.
    """
    if idxability != "Indexable":
        return 0

    h1_2_str = str(h1_2).strip()
    if h1_2_str != "" and h1_2_str != "nan":
        return 1  # second H1 exists

    return 0


def h2Eval(idxability, h2):
    """Checks H2 presence.
    Fails if H2 is missing, empty, or 'nan' (guards against .astype(str) coercion).
    """
    xRefResult = 0

    if idxability == "Indexable":
        h2_str = str(h2).strip()
        if h2_str == "" or h2_str == "nan" or len(h2_str) < 1:
            xRefResult = 1

    return xRefResult

def depthEval(depth, idxability):
    """Original depth evaluation function"""
    xRefResult = 0

    if idxability == "Indexable":
        if depth > 3:
            xRefResult = 1

    return xRefResult

def maxElems(idxability):
    """Original max elements function"""
    xRefResult = 6

    if idxability == "Indexable":
        xRefResult = 21

    return xRefResult

print("✅ Original SEO Analysis Functions loaded (including kwEval)")



In [ ]:
#@title Cell 7 - Updated SEO Analysis

# =============================================================================
# ENHANCED AI SEARCH ANALYSIS FUNCTIONS
# =============================================================================

def ai_structured_data_eval(schema_types, has_data, indexability):
    """AI structured data evaluation"""
    if indexability != "Indexable":
        return 0

    if not has_data:
        return 1  # Missing structured data is an issue

    # Check for AI-friendly schemas
    ai_schemas = ['Article', 'NewsArticle', 'BlogPosting', 'Product', 'Organization',
                 'Person', 'LocalBusiness', 'FAQPage', 'HowTo', 'QAPage']

    schema_list = str(schema_types).split(',') if schema_types else []
    has_ai_schema = any(schema in schema_list for schema in ai_schemas)

    return 0 if has_ai_schema else 1


def ai_heading_keyword_eval(h1, h2, brand_keywords, unbranded_keywords, indexability):
    """AI heading keyword optimization evaluation"""
    if indexability != "Indexable":
        return 0

    issues = 0
    h1_lower = str(h1).lower()
    h2_lower = str(h2).lower()

    all_keywords = brand_keywords + unbranded_keywords

    # Check H1 for keyword inclusion
    h1_has_keyword = any(kw.lower() in h1_lower for kw in all_keywords)
    if not h1_has_keyword:
        issues += 1

    # Check H2 for keyword inclusion (if H2 exists)
    if len(str(h2)) > 0:
        h2_has_keyword = any(kw.lower() in h2_lower for kw in all_keywords)
        if not h2_has_keyword:
            issues += 1

    return min(issues, 1)

def ai_content_structure_eval(semantic_count, list_count, indexability):
    """AI content structure evaluation"""
    if indexability != "Indexable":
        return 0

    issues = 0
    if semantic_count == 0:  # No semantic HTML
        issues += 1
    if list_count == 0:  # No lists for structure
        issues += 1

    return min(issues, 1)

def ai_answer_content_eval(question_count, indexability):
    """AI answer-ready content evaluation"""
    if indexability != "Indexable":
        return 0

    return 0 if question_count > 0 else 1

# NEW AI EVALUATION FUNCTIONS

def ai_citation_worthiness_eval(author_signals, citations, specific_claims, indexability):
    """AI citation-worthiness evaluation (NEW)"""
    if indexability != "Indexable":
        return 0

    issues = 0

    # Need author signals for credibility
    if author_signals == 0:
        issues += 1

    # Need external citations for authority
    if citations == 0:
        issues += 1

    # Need specific, factual content
    if specific_claims < 2:  # Less than 2 specific claims
        issues += 1

    return min(issues, 1)

def ai_multimodal_eval(video_count, video_captions, table_count, proper_tables, table_images, indexability):
    """AI multi-modal content evaluation (NEW)"""
    if indexability != "Indexable":
        return 0

    issues = 0

    # Videos without captions
    if video_count > 0 and video_captions / video_count < 0.8:
        issues += 1

    # Tables without proper markup
    if table_count > 0 and proper_tables / table_count < 0.8:
        issues += 1

    # Images of tables (should be HTML tables instead)
    if table_images > 0:
        issues += 1

    return min(issues, 1)

def ai_freshness_eval(total_date_signals, indexability):
    """AI content freshness evaluation (NEW)"""
    if indexability != "Indexable":
        return 0

    return 0 if total_date_signals >= 2 else 1

def ai_chunk_optimization_eval(words_per_heading, sections_with_context, total_sections, indexability):
    """AI chunk optimization evaluation (NEW)"""
    if indexability != "Indexable":
        return 0

    issues = 0

    # Too many words per heading (poor chunking)
    if words_per_heading > 300:
        issues += 1

    # Sections lack context
    if total_sections > 0 and sections_with_context / total_sections < 0.7:
        issues += 1

    return min(issues, 1)

def ai_bot_accessibility_eval(has_nosnippet, js_percentage, indexability):
    """AI bot accessibility evaluation (NEW)"""
    if indexability != "Indexable":
        return 0

    issues = 0

    # nosnippet blocks AI content usage
    if has_nosnippet:
        issues += 1

    # Too much JavaScript can prevent AI content extraction
    if js_percentage > 30:
        issues += 1

    return min(issues, 1)

def ai_maxElems(idxability):
    """AI max elements function (enhanced)"""
    return 9 if idxability == "Indexable" else 0  # 9 total AI factors



In [ ]:
#@title Cell 8 - AI scoring functions

# =============================================================================
# DISTINCT AI SCORING FUNCTIONS (NEW)
# =============================================================================

def calculate_distinct_ai_scores(crawl_frame):
    """Calculate distinct AI content vs technical scores"""

    eFrame = crawl_frame[crawl_frame["Indexability"] == "Indexable"]
    totalPages = len(eFrame)

    if totalPages == 0:
        return {
            'ai_content_score': 1.0,
            'ai_technical_score': 1.0,
            'ai_combined_score': 1.0,
            'content_issues': 0,
            'technical_issues': 0,
            'total_ai_issues': 0
        }

    # AI CONTENT FACTORS (5 factors - content structure & semantic markup)
    content_columns = [
        "AI Structured Data Issues",
        "AI Content Structure Issues",
        "AI Answer Content Issues",
        "AI Heading Keyword Issues",
        "AI Citation Worthiness Issues",
        "AI Freshness Issues"
    ]

    # AI TECHNICAL FACTORS (4 factors - crawlability & performance)
    technical_columns = [
        "AI Multi-Modal Issues",            # Video captions, table markup
        "AI Chunk Optimization Issues",     # Section independence & structure
        "AI Bot Accessibility Issues",      # JavaScript & nosnippet restrictions
        "AI Domain Crawlability Issues"     # Robots.txt AI bot blocking
    ]

    # Calculate issue totals
    content_issues = eFrame[content_columns].sum().sum()
    technical_issues = eFrame[technical_columns].sum().sum()
    total_ai_issues = content_issues + technical_issues

    # Calculate distinct scores
    ai_content_score = 1 - (content_issues / (totalPages * 6))  # 6 content factors
    ai_technical_score = 1 - (technical_issues / (totalPages * 4))  # 4 technical factors
    ai_combined_score = 1 - (total_ai_issues / (totalPages * 10))  # 10 total factors

    return {
        'ai_content_score': max(0, ai_content_score),
        'ai_technical_score': max(0, ai_technical_score),
        'ai_combined_score': max(0, ai_combined_score),
        'content_issues': content_issues,
        'technical_issues': technical_issues,
        'total_ai_issues': total_ai_issues,
        'total_pages': totalPages,
        'content_columns': content_columns,
        'technical_columns': technical_columns
    }

def add_distinct_ai_scores_to_dataframe(crawl_frame):
    """Add distinct AI score columns to the dataframe for Excel export"""

    # AI Content factors
    content_columns = [
        "AI Structured Data Issues",
        "AI Content Structure Issues",
        "AI Answer Content Issues",
        "AI Heading Keyword Issues",
        "AI Citation Worthiness Issues",
        "AI Freshness Issues"
    ]

    # AI Technical factors
    technical_columns = [
        "AI Multi-Modal Issues",
        "AI Chunk Optimization Issues",
        "AI Bot Accessibility Issues",
        "AI Domain Crawlability Issues"
    ]

    # Calculate page-level totals
    crawl_frame["AI Content Issues Total"] = crawl_frame[content_columns].sum(axis=1)
    crawl_frame["AI Technical Issues Total"] = crawl_frame[technical_columns].sum(axis=1)
    crawl_frame["AI Combined Issues Total"] = crawl_frame[content_columns + technical_columns].sum(axis=1)

    # Calculate page-level scores (0-1 for each page)
    crawl_frame["AI Content Score"] = crawl_frame.apply(
        lambda x: 1 - (x["AI Content Issues Total"] / 6) if x["Indexability"] == "Indexable" else 1, axis=1
    )
    crawl_frame["AI Technical Score"] = crawl_frame.apply(
        lambda x: 1 - (x["AI Technical Issues Total"] / 4) if x["Indexability"] == "Indexable" else 1, axis=1
    )
    crawl_frame["AI Combined Score"] = crawl_frame.apply(
        lambda x: 1 - (x["AI Combined Issues Total"] / 10) if x["Indexability"] == "Indexable" else 1, axis=1
    )

    return crawl_frame

def generate_enhanced_report_with_distinct_scores(crawl_frame, col_list, ai_col_list, robots_data):
    """Generate enhanced report with distinct AI content vs technical scoring"""

    # Calculate distinct AI scores
    ai_scores = calculate_distinct_ai_scores(crawl_frame)

    eFrame = crawl_frame[crawl_frame["Indexability"] == "Indexable"]
    totalErrors = eFrame["Total SEO Errors"].sum()
    totalPages = eFrame.shape[0]

    print("="*70)
    print("ENHANCED COMPREHENSIVE AUDIT RESULTS WITH DISTINCT AI SCORING")
    print("="*70)

    print("\\nTRADITIONAL SEO ANALYSIS")
    print("-" * 30)
    print(f"Total URLs Analyzed: {len(crawl_frame)}")
    print(f"Indexable Pages: {totalPages}")
    print(f"Traditional SEO Errors: {totalErrors}")
    print(f"SEO Errors per page: {totalErrors/totalPages:.1f}" if totalPages > 0 else "SEO Errors per page: 0")
    print(f"Traditional SEO Score: {1-(totalErrors/(totalPages*21)):.3f}" if totalPages > 0 else "Traditional SEO Score: 1.000")

    print("\\nDISTINCT AI SEARCH SCORING")
    print("-" * 30)
    print(f"AI Content Readiness Score:    {ai_scores['ai_content_score']:.3f} ({ai_scores['ai_content_score']*100:.1f}%)")
    print(f"AI Technical Readiness Score:  {ai_scores['ai_technical_score']:.3f} ({ai_scores['ai_technical_score']*100:.1f}%)")
    print(f"AI Combined Readiness Score:   {ai_scores['ai_combined_score']:.3f} ({ai_scores['ai_combined_score']*100:.1f}%)")

    print("\\nAI ISSUE BREAKDOWN")
    print("-" * 20)
    print(f"AI Content Issues:    {ai_scores['content_issues']:3d} / {ai_scores['total_pages']*5:3d} possible")
    print(f"AI Technical Issues:  {ai_scores['technical_issues']:3d} / {ai_scores['total_pages']*4:3d} possible")
    print(f"Total AI Issues:      {ai_scores['total_ai_issues']:3d} / {ai_scores['total_pages']*9:3d} possible")

    print("\\nAI CONTENT FACTOR BREAKDOWN")
    print("-" * 30)
    for col in ai_scores['content_columns']:
        issue_count = eFrame[col].sum()
        factor_name = col.replace("AI ", "").replace(" Issues", "")
        print(f"{factor_name:25s}: {issue_count:3d}")

    print("\\nAI TECHNICAL FACTOR BREAKDOWN")
    print("-" * 32)
    for col in ai_scores['technical_columns']:
        issue_count = eFrame[col].sum()
        factor_name = col.replace("AI ", "").replace(" Issues", "")
        print(f"{factor_name:25s}: {issue_count:3d}")

    print("\\nSTRATEGIC AI OPTIMIZATION RECOMMENDATIONS")
    print("-" * 42)

    content_score = ai_scores['ai_content_score']
    technical_score = ai_scores['ai_technical_score']

    if content_score >= 0.8 and technical_score >= 0.8:
        print("🟢 EXCELLENT: Both content and technical AI optimization are strong")
        print("   → Monitor performance and maintain current optimization")

    elif content_score >= 0.8 and technical_score < 0.6:
        print("🔵 FOCUS ON AI TECHNICAL: Content optimization is strong")
        print("   → Priority 1: Fix AI bot crawlability issues")
        print("   → Priority 2: Improve multi-modal content (video captions, table markup)")
        print("   → Priority 3: Optimize content chunking and structure")

    elif content_score < 0.6 and technical_score >= 0.8:
        print("🟡 FOCUS ON AI CONTENT: Technical foundation is strong")
        print("   → Priority 1: Add structured data (JSON-LD) to pages")
        print("   → Priority 2: Implement author markup and citations")
        print("   → Priority 3: Add publication dates and freshness signals")
        print("   → Priority 4: Create FAQ and answer-ready content")

    elif content_score < 0.6 and technical_score < 0.6:
        print("🔴 COMPREHENSIVE AI OVERHAUL: Both areas need significant improvement")
        print("   → Phase 1: Fix AI content structure (faster ROI)")
        print("   → Phase 2: Address technical crawlability issues")
        print("   → Phase 3: Implement comprehensive AI optimization strategy")

    else:
        print("🟡 BALANCED AI IMPROVEMENT: Moderate enhancement needed in both areas")
        print("   → Address highest-impact issues in both content and technical")
        print("   → Focus on quick wins: structured data + bot accessibility")

    print("\\nAI OPTIMIZATION GRADES")
    print("-" * 22)

    def get_grade(score):
        if score >= 0.9: return "A+"
        elif score >= 0.8: return "A"
        elif score >= 0.7: return "B"
        elif score >= 0.6: return "C"
        elif score >= 0.5: return "D"
        else: return "F"

    print(f"AI Content Grade:     {get_grade(content_score)} ({content_score:.3f})")
    print(f"AI Technical Grade:   {get_grade(technical_score)} ({technical_score:.3f})")
    print(f"AI Overall Grade:     {get_grade(ai_scores['ai_combined_score'])} ({ai_scores['ai_combined_score']:.3f})")

    # Traditional SEO breakdown (keeping existing functionality)
    print("\\nTRADITIONAL SEO ISSUE BREAKDOWN")
    print("-" * 35)
    shortErrors = (eFrame["Does Title Exist"].sum() + eFrame["Title Brand Relevance"].sum() +
                  eFrame["Title Unbranded Relevance"].sum() + eFrame["Does MDesc Exist"].sum())

    midErrors = (eFrame["Page Indexability"].sum() + eFrame["HTTP Responses"].sum() +
                eFrame["MDesc Brand Relevance"].sum() + eFrame["MDesc Unbranded Relevance"].sum())

    longErrors = (eFrame["Is Internal Linking Optimized"].sum() + eFrame["Sufficient Body Content?"].sum() +
                 eFrame["Is Experience Optimized?"].sum() + eFrame["Are URLs Optimized"].sum())

    print(f"Low Hanging Fruit: {shortErrors}")
    print(f"Mid Level Issues: {midErrors}")
    print(f"Long Term Problems: {longErrors}")

    return ai_scores



In [ ]:
#@title Cell 9 - Main Execution Function (Updated for SimplifiedCrawler)

# =============================================================================
# UPDATED MAIN EXECUTION SECTION FOR SIMPLIFIED CRAWLER
# =============================================================================

def run_traditional_crawl_audit_with_distinct_scoring():
    """Run traditional crawl audit with distinct scoring using SimplifiedCrawler"""

    print("="*70)
    print("SIMPLIFIED WEB CRAWLER WITH ENHANCED SEO + AI AUDIT")
    print("="*70)
    print(f"Starting domain: {domain}")
    print(f"Max pages: {MaxPages}")
    print(f"Using requests-based crawler (simulates AI bot behavior)")

    # Analyze AI bot crawlability at domain level
    print("\nAnalyzing AI bot crawlability...")
    robots_data = analyze_ai_bot_crawlability(domain)

    if robots_data['robots_txt_exists']:
        print(f"✅ Robots.txt found")
        if robots_data['total_ai_bots_blocked'] > 0:
            print(f"⚠️  {robots_data['total_ai_bots_blocked']} AI bots blocked: {', '.join(robots_data['ai_bots_blocked'])}")
        else:
            print("✅ No AI bots explicitly blocked")
    else:
        print("⚠️  No robots.txt found")

    # Initialize simplified crawler
    crawler = SimplifiedCrawler(
        domain=domain,
        max_pages=MaxPages,
        crawl_delay=CrawlDelay,
        pagespeed_api_key=pagespeed_api_key,
        sample_size=pagespeed_sample_size
    )

    try:
        # Progress callback
        def progress_callback(current, total, url):
            if current % 5 == 0 or current == total:
                print(f"[{current:3d}/{total:3d}] Crawling: {url[:60]}...")

        # Start traditional crawl from homepage
        print(f"\nStarting traditional crawl from homepage...")
        crawlFrame = crawler.traditional_crawl(domain, progress_callback)

        # Get crawl statistics
        stats = crawler.get_crawl_stats()
        print(f"\nCrawl Statistics:")
        print(f"  Pages successfully crawled: {stats['pages_crawled']}")
        print(f"  Pages failed: {stats['pages_failed']}")
        print(f"  Total URLs discovered: {stats['urls_discovered']}")
        print(f"  URLs remaining in queue: {stats['queue_remaining']}")
        print(f"  Crawl efficiency: {stats['crawl_efficiency']:.2%}")

        print(f"\nAnalyzing {len(crawlFrame)} pages...")

        # Data preprocessing (from original tool)
        crawlFrame["Content Type"] = crawlFrame["Content Type"].astype(str)
        crawlFrame = crawlFrame[crawlFrame['Content Type'].str.contains("text/html")]

        addresses = crawlFrame["Address"].values.tolist()

        # Data type conversions (from original tool)
        crawlFrame["Title 1"] = crawlFrame["Title 1"].astype(str)
        crawlFrame["Meta Description 1"] = crawlFrame["Meta Description 1"].astype(str)
        crawlFrame["Canonical Link Element 1"] = crawlFrame["Canonical Link Element 1"].astype(str)
        crawlFrame["Address"] = crawlFrame["Address"].astype(str)
        crawlFrame["H1-1"] = crawlFrame["H1-1"].astype(str)
        crawlFrame["H1-2"] = crawlFrame["H1-2"].astype(str)
        crawlFrame["H2-1"] = crawlFrame["H2-1"].astype(str)

        # Performance calculations (from original tool)
        perfScore = crawlFrame["Performance Score"].mean()
        avgLCP = crawlFrame["CrUX Largest Contentful Paint Time (ms)"].mean()
        avgCLS = crawlFrame["CrUX Cumulative Layout Shift"].mean()
        avgFID = crawlFrame["CrUX Interaction to Next Paint (ms)"].mean()

        crawlFrame["Performance Score"] = crawlFrame["Performance Score"].fillna(perfScore)
        crawlFrame["CrUX Largest Contentful Paint Time (ms)"] = crawlFrame["CrUX Largest Contentful Paint Time (ms)"].fillna(avgLCP)
        crawlFrame["CrUX Cumulative Layout Shift"] = crawlFrame["CrUX Cumulative Layout Shift"].fillna(avgCLS)
        crawlFrame["CrUX Interaction to Next Paint (ms)"] = crawlFrame["CrUX Interaction to Next Paint (ms)"].fillna(avgFID)

        # -----------------------------------------------------------------------
        # SITEMAP: fetch XML sitemap and tag each page
        # -----------------------------------------------------------------------
        sitemap_urls = set()
        if siteMapAddress and siteMapAddress.strip():
            print(f"\nFetching XML sitemap: {siteMapAddress}")
            try:
                import xml.etree.ElementTree as ET
                sm_response = requests.get(siteMapAddress.strip(), timeout=15, headers={
                    'User-Agent': 'Mozilla/5.0'
                })
                if sm_response.status_code == 200:
                    root = ET.fromstring(sm_response.content)
                    ns = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
                    # Handle both sitemap index and regular sitemaps
                    locs = root.findall('.//sm:loc', ns) or root.findall('.//loc')
                    for loc in locs:
                        url = loc.text.strip().rstrip('/') if loc.text else ''
                        if url:
                            sitemap_urls.add(url)
                    print(f"  ✅ Loaded {len(sitemap_urls)} URLs from sitemap")
                else:
                    print(f"  ⚠️  Sitemap returned status {sm_response.status_code} — skipping")
            except Exception as sm_err:
                print(f"  ⚠️  Could not parse sitemap: {sm_err} — sitemap scoring disabled")
        else:
            print("\nNo sitemap URL provided — sitemap scoring disabled")

        crawlFrame["In Sitemap"] = crawlFrame["Address"].apply(
            lambda url: "yes" if str(url).rstrip('/') in sitemap_urls else "no"
        )
        sitemap_list = list(sitemap_urls)

        print("Applying traditional SEO analysis...")
        print(f"  Active brand keywords:     {branKeywords}")
        print(f"  Active unbranded keywords: {unbKeywords}")

        # Apply original SEO analysis functions
        crawlFrame["Does Title Exist"] = crawlFrame.apply(lambda x: titleEval(x["Indexability"], x["Title 1"]), axis=1)
        crawlFrame["Is Title Length Optimized"] = crawlFrame.apply(lambda x: titleLengthEval(x["Indexability"], x["Title 1"], length_thresholds), axis=1)
        crawlFrame["Title Brand Relevance"] = crawlFrame.apply(lambda x: kwEval(x["Indexability"], x["Title 1"], branKeywords), axis=1)
        crawlFrame["Title Unbranded Relevance"] = crawlFrame.apply(lambda x: kwEval(x["Indexability"], x["Title 1"], unbKeywords), axis=1)
        crawlFrame["HTTP Responses"] = crawlFrame.apply(lambda x: httpEval(x["Status Code"], x.get("Unique Inlinks", 0)), axis=1)
        crawlFrame["Does MDesc Exist"] = crawlFrame.apply(lambda x: mdEval(x["Indexability"], x["Meta Description 1"]), axis=1)
        crawlFrame["Is MDesc Length Optimized"] = crawlFrame.apply(lambda x: mdLengthEval(x["Indexability"], x["Meta Description 1"], length_thresholds), axis=1)
        crawlFrame["MDesc Brand Relevance"] = crawlFrame.apply(lambda x: kwEval(x["Indexability"], x["Meta Description 1"], branKeywords), axis=1)
        crawlFrame["MDesc Unbranded Relevance"] = crawlFrame.apply(lambda x: kwEval(x["Indexability"], x["Meta Description 1"], unbKeywords), axis=1)
        crawlFrame["Are Canonicals Optimized?"] = crawlFrame.apply(lambda x: canonicalEval(x["Status Code"], x["Canonical Link Element 1"], x["Address"], addresses, sitemap_list if sitemap_list else addresses, x["Indexability"], x.get("Canonical Count", 1)), axis=1)
        crawlFrame["Are There Sitemap Errors?"] = crawlFrame.apply(lambda x: siteMapEval(x["In Sitemap"], x["Indexability"]), axis=1)
        crawlFrame["Is Internal Linking Optimized"] = crawlFrame.apply(lambda x: internalLinkEval(x["Unique Inlinks"], x["Indexability"], MaxPages), axis=1)
        crawlFrame["Sufficient Body Content?"] = crawlFrame.apply(lambda x: contentEval(x["Word Count"], x["Indexability"]), axis=1)
        crawlFrame["Does H1 Exist"] = crawlFrame.apply(lambda x: headingEval(x["H1-1"], x["Indexability"]), axis=1)
        crawlFrame["Multiple H1 Tags"] = crawlFrame.apply(lambda x: multipleH1Eval(x["Indexability"], x["H1-2"]), axis=1)
        crawlFrame["Does H2 Exist"] = crawlFrame.apply(lambda x: h2Eval(x["Indexability"], x["H2-1"]), axis=1)
        crawlFrame["H1 Keyword Relevance"] = crawlFrame.apply(lambda x: h1KeywordEval(x["Indexability"], x["H1-1"], branKeywords, unbKeywords), axis=1)
        crawlFrame["Are URLs Optimized"] = crawlFrame.apply(lambda x: addressEval(x["Address"], x["Indexability"]), axis=1)
        crawlFrame["Is Experience Optimized?"] = crawlFrame.apply(lambda x: speedEval(x["CrUX Largest Contentful Paint Time (ms)"], x["CrUX Cumulative Layout Shift"], x["CrUX Interaction to Next Paint (ms)"], x["Indexability"]), axis=1)
        crawlFrame["Are titles duplicated?"] = crawlFrame.apply(lambda x: duplicationEval(x["Title 1"], crawlFrame["Title 1"].values.tolist(), x["Indexability"]), axis=1)
        crawlFrame["Are descriptions duplicated?"] = crawlFrame.apply(lambda x: duplicationEval(x["Meta Description 1"], crawlFrame["Meta Description 1"].values.tolist(), x["Indexability"]), axis=1)
        crawlFrame["Are H1s duplicated?"] = crawlFrame.apply(lambda x: duplicationEval(x["H1-1"], crawlFrame["H1-1"].values.tolist(), x["Indexability"]), axis=1)
        # "Are subheads missing?" retired — H2 now evaluated as "Does H2 Exist" above
        crawlFrame["Page Indexability"] = crawlFrame.apply(lambda x: indexEval(x["Indexability"], x["Status Code"]), axis=1)
        crawlFrame["Crawl Depth?"] = crawlFrame.apply(lambda x: depthEval(x["Crawl Depth"], x["Indexability"]), axis=1)
        crawlFrame["HTML Size Issues"] = crawlFrame.apply(lambda x: htmlSizeEval(x.get("HTML Size (bytes)", 0), x["Indexability"]), axis=1)

        crawlFrame["Max Elements"] = crawlFrame.apply(lambda x: maxElems(x["Indexability"]), axis=1)

        print("Applying enhanced AI search analysis...")

        # Apply original AI search analysis functions
        crawlFrame["AI Structured Data Issues"] = crawlFrame.apply(
            lambda x: ai_structured_data_eval(x["AI_Schema_Types"], x["AI_Has_Structured_Data"], x["Indexability"]), axis=1
        )
        crawlFrame["AI Content Structure Issues"] = crawlFrame.apply(
            lambda x: ai_content_structure_eval(x["AI_Semantic_Elements"], x["AI_List_Elements"], x["Indexability"]), axis=1
        )
        crawlFrame["AI Answer Content Issues"] = crawlFrame.apply(
            lambda x: ai_answer_content_eval(x["AI_Question_Content"], x["Indexability"]), axis=1
        )

        # Apply NEW AI search analysis functions
        crawlFrame["AI Heading Keyword Issues"] = crawlFrame.apply(
            lambda x: ai_heading_keyword_eval(x["H1-1"], x["H2-1"], branKeywords, unbKeywords, x["Indexability"]), axis=1
        )
        crawlFrame["AI Citation Worthiness Issues"] = crawlFrame.apply(
            lambda x: ai_citation_worthiness_eval(x["AI_Author_Signals"], x["AI_External_Citations"], x["AI_Specific_Claims"], x["Indexability"]), axis=1
        )
        crawlFrame["AI Multi-Modal Issues"] = crawlFrame.apply(
            lambda x: ai_multimodal_eval(x["AI_Video_Count"], x["AI_Videos_With_Captions"], x["AI_Table_Count"], x["AI_Proper_Table_Markup"], x["AI_Table_Images_Count"], x["Indexability"]), axis=1
        )
        crawlFrame["AI Freshness Issues"] = crawlFrame.apply(
            lambda x: ai_freshness_eval(x["AI_Total_Date_Signals"], x["Indexability"]), axis=1
        )
        crawlFrame["AI Chunk Optimization Issues"] = crawlFrame.apply(
            lambda x: ai_chunk_optimization_eval(x["AI_Words_Per_Heading"], x["AI_Sections_With_Context"], x["AI_Total_Sections"], x["Indexability"]), axis=1
        )
        crawlFrame["AI Bot Accessibility Issues"] = crawlFrame.apply(
            lambda x: ai_bot_accessibility_eval(x["AI_Has_NoSnippet"], x["AI_JS_Percentage"], x["Indexability"]), axis=1
        )

        # Add domain-level AI bot crawlability issues
        crawlFrame["AI Domain Crawlability Issues"] = robots_data['total_ai_bots_blocked'] > 0

        crawlFrame["AI Max Elements"] = crawlFrame.apply(lambda x: ai_maxElems(x["Indexability"]), axis=1)

        # Calculate error totals
        # -----------------------------------------------------------------------
        # CASCADE FAIL: auto-fail all SEO checks for broken/redirect pages
        # -----------------------------------------------------------------------
        crawlFrame["Should Cascade Fail"] = crawlFrame.apply(
            lambda x: crawlCascadeFail(x["Status Code"], x.get("Unique Inlinks", 0), x["In Sitemap"]), axis=1
        )

        # Override individual eval columns to 1 for cascade-fail pages
        cascade_override_cols = [
            "Does Title Exist", "Is Title Length Optimized", "Title Brand Relevance", "Title Unbranded Relevance",
            "Does MDesc Exist", "Is MDesc Length Optimized", "MDesc Brand Relevance", "MDesc Unbranded Relevance",
            "Are Canonicals Optimized?", "Is Internal Linking Optimized",
            "Sufficient Body Content?", "Does H1 Exist", "Multiple H1 Tags", "Does H2 Exist", "H1 Keyword Relevance",
            "Are URLs Optimized", "Is Experience Optimized?",
            "Are titles duplicated?", "Are descriptions duplicated?", "Are H1s duplicated?",
            "Crawl Depth?", "HTML Size Issues"
        ]
        for col in cascade_override_cols:
            if col in crawlFrame.columns:
                crawlFrame.loc[crawlFrame["Should Cascade Fail"], col] = 1

        cascade_fail_count = crawlFrame["Should Cascade Fail"].sum()
        if cascade_fail_count > 0:
            print(f"  ⚠️  Cascade fail applied to {cascade_fail_count} pages (broken status, temp redirects, heavily-linked 301s, or sitemap/status mismatches)")

        col_list = ["Does Title Exist","Is Title Length Optimized","Title Brand Relevance","Title Unbranded Relevance","HTTP Responses","Does MDesc Exist","Is MDesc Length Optimized","MDesc Brand Relevance",
                   "MDesc Unbranded Relevance","Are Canonicals Optimized?","Are There Sitemap Errors?","Is Internal Linking Optimized","Sufficient Body Content?",
                   "Does H1 Exist","Multiple H1 Tags","Does H2 Exist","H1 Keyword Relevance","Are URLs Optimized","Is Experience Optimized?","Are titles duplicated?",
                   "Are descriptions duplicated?","Are H1s duplicated?","Page Indexability","Crawl Depth?","HTML Size Issues"]

        # Enhanced AI column list with all factors
        ai_col_list = ["AI Structured Data Issues", "AI Content Structure Issues", "AI Answer Content Issues",
                      "AI Citation Worthiness Issues", "AI Multi-Modal Issues", "AI Freshness Issues",
                      "AI Chunk Optimization Issues", "AI Bot Accessibility Issues", "AI Domain Crawlability Issues"]

        crawlFrame["Total SEO Errors"] = crawlFrame[col_list].sum(axis=1)
        crawlFrame["Total AI Issues"] = crawlFrame[ai_col_list].sum(axis=1)
        crawlFrame["Total Enhanced Errors"] = crawlFrame[col_list + ai_col_list].sum(axis=1)

        crawlFrame["Error Rate"] = crawlFrame["Total SEO Errors"]/crawlFrame["Max Elements"]
        crawlFrame["AI Error Rate"] = crawlFrame["Total AI Issues"]/crawlFrame["AI Max Elements"].replace(0, 1)
        crawlFrame["Enhanced Error Rate"] = crawlFrame["Total Enhanced Errors"]/(crawlFrame["Max Elements"] + crawlFrame["AI Max Elements"])

        print("\nAdding distinct AI scoring...")

        # Add distinct AI scores to dataframe
        crawlFrame = add_distinct_ai_scores_to_dataframe(crawlFrame)

        print("\nGenerating enhanced comprehensive report with distinct scoring...")

        # Generate enhanced report with distinct scores
        ai_scores = generate_enhanced_report_with_distinct_scores(crawlFrame, col_list, ai_col_list, robots_data)

        # Export results
        print("\nExporting enhanced results with distinct scoring...")
        crawlFrame.to_excel(nameOfNewSpreadSheet, index=None, engine='openpyxl')

        print(f"\n✅ Enhanced results with distinct AI scoring exported to: {nameOfNewSpreadSheet}")
        print("\nExcel file now includes:")
        print("  → Traditional SEO scoring (21 factors)")
        print("  → AI Content Score (per page)")
        print("  → AI Technical Score (per page)")
        print("  → AI Combined Score (per page)")
        print("  → Strategic recommendations based on distinct scores")
        print("  → Requests-based crawl (simulates AI bot behavior)")
        if pagespeed_api_key:
            print(f"  → Real PageSpeed data for {pagespeed_sample_size} sample pages")

        return crawlFrame, ai_scores

    except Exception as e:
        print(f"\n❌ Error during crawl audit: {e}")
        import traceback
        traceback.print_exc()
        raise

print("✅ Main execution function loaded (using SimplifiedCrawler)")



In [ ]:
#@title Cell 10 - Run main script

if __name__ == "__main__":
    print("Starting Traditional Web Crawler with Enhanced SEO + AI Search Audit...")
    print("Includes comprehensive factors from Aleyda Solis's AI Search Optimization Checklist")
    print("NOW WITH TRADITIONAL CRAWLING + DISTINCT AI CONTENT vs TECHNICAL SCORING!")

    try:
        results, ai_scores = run_traditional_crawl_audit_with_distinct_scoring()
        print("\n" + "="*70)
        print("TRADITIONAL CRAWLER AUDIT WITH DISTINCT SCORING COMPLETED!")
        print("="*70)
        print(f"Results saved to: {nameOfNewSpreadSheet}")
        print("\nYour traditional crawl audit now includes:")
        print("✅ Traditional web crawling (starts at homepage, discovers links)")
        print("✅ Traditional SEO optimization (21 factors)")
        print("✅ AI Content optimization scoring (6 factors)")
        print("✅ AI Technical optimization scoring (4 factors)")
        print("✅ Combined AI readiness scoring (10 factors)")
        print("✅ Strategic recommendations based on distinct AI scores")
        print("✅ Citation-worthiness analysis")
        print("✅ Multi-modal content optimization")
        print("✅ Content freshness signals")
        print("✅ Chunk-level optimization")
        print("✅ AI bot crawlability analysis")
        print("✅ Crawl statistics and efficiency metrics")
        print("✅ Total: 31 comprehensive optimization factors!")

        print(f"\n🎯 FINAL SCORES:")
        indexable_pages = results[results['Indexability']=='Indexable']
        if len(indexable_pages) > 0:
            traditional_score = 1-(indexable_pages['Total SEO Errors'].sum()/(len(indexable_pages)*21))
            print(f"Traditional SEO Score: {traditional_score:.3f}")
        else:
            print("Traditional SEO Score: 1.000")

        print(f"AI Content Score: {ai_scores['ai_content_score']:.3f}")
        print(f"AI Technical Score: {ai_scores['ai_technical_score']:.3f}")
        print(f"AI Combined Score: {ai_scores['ai_combined_score']:.3f}")

        # Show crawl statistics
        print(f"\n📊 CRAWL STATISTICS:")
        print(f"Pages Successfully Crawled: {len(results)}")
        print(f"Indexable Pages Found: {len(indexable_pages)}")
        print(f"Average Links per Page: {results['Links Discovered'].mean():.1f}")

        # Prepare summary data for email
        def get_grade(score):
            if score >= 0.9: return "A+"
            elif score >= 0.8: return "A"
            elif score >= 0.7: return "B"
            elif score >= 0.6: return "C"
            elif score >= 0.5: return "D"
            else: return "F"

        email_ai_scores = {
            'traditional_seo_score': traditional_score if len(indexable_pages) > 0 else 1.0,
            'total_seo_errors': int(indexable_pages['Total SEO Errors'].sum()) if len(indexable_pages) > 0 else 0,
            'errors_per_page': indexable_pages['Total SEO Errors'].sum() / len(indexable_pages) if len(indexable_pages) > 0 else 0,
            'ai_content_score': ai_scores['ai_content_score'],
            'ai_technical_score': ai_scores['ai_technical_score'],
            'ai_combined_score': ai_scores['ai_combined_score'],
            'content_issues': ai_scores['content_issues'],
            'technical_issues': ai_scores['technical_issues'],
            'total_ai_issues': ai_scores['total_ai_issues'],
            'total_pages': ai_scores['total_pages'],
            'content_grade': get_grade(ai_scores['ai_content_score']),
            'technical_grade': get_grade(ai_scores['ai_technical_score']),
            'combined_grade': get_grade(ai_scores['ai_combined_score'])
        }

        email_crawl_stats = {
            'total_pages': len(results),
            'indexable_pages': len(indexable_pages),
            'avg_links': f"{results['Links Discovered'].mean():.1f}"
        }

        send_email_report(email_address, nameOfNewSpreadSheet, domain, email_ai_scores, email_crawl_stats)
    except Exception as e:
        print(f"\nError during traditional crawl audit: {e}")
        print("Please check your configuration and try again.")

    print("\nTraditional crawler audit process finished.")

In [ ]:
#@title Cell 11 - Screaming Frog Upload (Fixed Column Handling)

from google.colab import files
import io

sf_dataframe = None

print("="*70)
print("SCREAMING FROG IMPORT MODE")
print("="*70)
print("\nThis mode analyzes a Screaming Frog export instead of live crawling.")
print("Benefits:")
print("  ✓ No 406/403 errors")
print("  ✓ Works with protected sites")
print("  ✓ Faster for large sites")
print("  ✓ JavaScript rendering support")
print("  ✓ Can handle authenticated pages")
print("="*70)

# ============================================================================
# USE EXISTING CONFIGURATION FROM CELL 3
# ============================================================================

print("\n📋 USING CONFIGURATION FROM CELL 3")
print("-"*70)

try:
    print(f"Domain: {AnalyzedDomain}")
    print(f"Brand Keywords: {BrandKeywords}")
    print(f"Unbranded Keywords: {UnbrandedKeywords}")

    sf_domain = AnalyzedDomain
    if not sf_domain.startswith('http'):
        sf_domain = 'https://' + sf_domain
    sf_domain_parsed = urlparse(sf_domain).netloc

    print("✓ Configuration loaded from Cell 3")

except NameError as e:
    print("❌ ERROR: Cell 3 variables not found.")
    print(f"   Missing: {e}")
    raise

print("-"*70)

# ============================================================================
# FILE UPLOAD
# ============================================================================

print("\n" + "="*70)
print("📤 UPLOAD SCREAMING FROG EXPORT")
print("="*70)
print("\n")

uploaded = files.upload()

sf_raw_data = None
if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\n✓ File uploaded: {filename}")

    try:
        if filename.lower().endswith('.csv'):
            sf_raw_data = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8')
        elif filename.lower().endswith(('.xlsx', '.xls')):
            sf_raw_data = pd.read_excel(io.BytesIO(uploaded[filename]))

        if sf_raw_data is not None:
            print(f"✓ Data loaded: {len(sf_raw_data):,} rows, {len(sf_raw_data.columns)} columns")

    except Exception as e:
        print(f"❌ Error: {e}")
        sf_raw_data = None

print("\n" + "="*70)

# ============================================================================
# PARSE DATA - KEEP ALL COLUMNS
# ============================================================================

if sf_raw_data is not None:
    print("="*70)
    print("PARSING SCREAMING FROG DATA")
    print("="*70)

    # Find required columns
    def find_column(df, possible_names):
        for name in possible_names:
            if name in df.columns:
                return name
        return None

    url_col = find_column(sf_raw_data, ['Address', 'URL'])
    status_col = find_column(sf_raw_data, ['Status Code', 'Status'])
    index_col = find_column(sf_raw_data, ['Indexability'])

    if not all([url_col, status_col, index_col]):
        print("❌ Missing required columns")
        sf_dataframe = None
    else:
        print(f"\n✓ Found required columns")
        print(f"  URL: {url_col}")
        print(f"  Status: {status_col}")
        print(f"  Indexability: {index_col}")

        # Filter to internal URLs
        print(f"\n📊 Filtering to internal URLs...")
        internal_mask = sf_raw_data[url_col].apply(
            lambda url: sf_domain_parsed in str(urlparse(str(url)).netloc) if pd.notna(url) else False
        )

        sf_dataframe = sf_raw_data[internal_mask].copy()
        print(f"✓ Kept {len(sf_dataframe):,} internal URLs")

        # Create standardized column mapping WITHOUT duplicating
        rename_dict = {}

        # Only rename if the standardized name doesn't already exist
        if url_col != 'URL' and 'URL' not in sf_dataframe.columns:
            rename_dict[url_col] = 'URL'
        if status_col != 'Status Code' and 'Status Code' not in sf_dataframe.columns:
            rename_dict[status_col] = 'Status Code'
        if index_col != 'Indexability' and 'Indexability' not in sf_dataframe.columns:
            rename_dict[index_col] = 'Indexability'

        # Add other common renames (only if target doesn't exist)
        potential_renames = {
            'Title 1': 'Title',
            'Meta Description 1': 'Meta Description',
            'H1-1': 'H1',
            'H2-1': 'H2',
            'Canonical Link Element 1': 'Canonical',
            'Meta Robots 1': 'Meta Robots',
            'X-Robots-Tag 1': 'X-Robots-Tag',
            'Unique Inlinks': 'Inlinks',
            'Unique Outlinks': 'Outlinks',
            'Largest Contentful Paint Time (ms)': 'LCP',
            'Cumulative Layout Shift': 'CLS',
            'Interaction to Next Paint (ms)': 'INP'
        }

        for old_name, new_name in potential_renames.items():
            if old_name in sf_dataframe.columns and new_name not in sf_dataframe.columns:
                rename_dict[old_name] = new_name

        if rename_dict:
            sf_dataframe = sf_dataframe.rename(columns=rename_dict)
            print(f"\n✓ Standardized {len(rename_dict)} column names")

        # Add missing essential columns
        essential_defaults = {
            'Title': '',
            'Meta Description': '',
            'H1': '',
            'H2': '',
            'Word Count': 0,
            'Inlinks': 0,
            'Outlinks': 0,
            'Canonical': '',
            'LCP': 0,
            'CLS': 0,
            'INP': 0,
            'In Sitemap': 'Unknown'
        }

        added_cols = []
        for col, default in essential_defaults.items():
            if col not in sf_dataframe.columns:
                sf_dataframe[col] = default
                added_cols.append(col)

        if added_cols:
            print(f"✓ Added {len(added_cols)} default columns: {', '.join(added_cols[:5])}")

        # Add Address if not present (duplicate of URL)
        if 'Address' not in sf_dataframe.columns:
            sf_dataframe['Address'] = sf_dataframe['URL']

        print

In [ ]:
#@title Cell 12: Analyze SF Data with Scoring Logic

# ====================================================================
# CELL 12: ANALYZE SCREAMING FROG DATA WITH COMPLETE SCORING LOGIC
# ====================================================================

import pandas as pd
import numpy as np
from difflib import SequenceMatcher

print("="*70)
print("🔍 GENERATING SEO + AI SEARCH ANALYSIS")
print("="*70)

# Check if SF data was imported
if 'sf_dataframe' not in dir():
    print("\n❌ ERROR: No Screaming Frog data found.")
    print("   Please run Cell 11 first to import your CSV file.")
    raise Exception("Screaming Frog data not loaded")

print(f"\nAnalyzing {len(sf_dataframe):,} pages...")
print(f"Domain: {AnalyzedDomain if 'AnalyzedDomain' in dir() else 'Unknown'}")
print(f"Using keywords from Cell 3:")
print(f"  Brand: {BrandKeywords if 'BrandKeywords' in dir() else 'Not set'}")
print(f"  Unbranded: {UnbrandedKeywords if 'UnbrandedKeywords' in dir() else 'Not set'}")
print("="*70)

# ====================================================================
# DEFINE EVALUATION FUNCTIONS
# ====================================================================

def page_indexability_eval(indexability, status_code, in_sitemap, inlinks):
    """
    Flags non-indexable pages that SHOULD be indexable based on:
    1. Listed in XML sitemap (indicates intent to index)
    2. Has 8+ internal links (indicates importance)
    """
    if indexability == "Non-Indexable" and status_code == 200:
        # Flag if in sitemap
        if str(in_sitemap).lower() == 'yes':
            return 1

        # Flag if heavily linked (8+ inlinks)
        try:
            if int(inlinks) >= 8:
                return 1
        except:
            pass

    return 0

def http_response_eval(status_code, inlinks):
    """
    Evaluates HTTP status with link equity consideration.

    Returns 1 (issue) if:
    - Status is not 200 or 301
    - Status is 301 BUT page has 10+ inlinks (architectural issue)
    """
    try:
        code = int(status_code)
        inlink_count = int(inlinks)

        # 200 is always good
        if code == 200:
            return 0

        # 301 is acceptable UNLESS heavily linked
        if code == 301:
            if inlink_count >= 10:
                return 1  # Heavily linked 301 = architectural problem
            return 0  # Low-linked 301 = acceptable

        # Everything else is bad (302, 404, 500, etc.)
        return 1

    except:
        return 1  # Error parsing = issue

def should_cascade_fail(indexability_issue, http_issue, status_code, inlinks):
    """
    Determine if page should auto-fail all SEO tests.

    Returns True if:
    1. Page has indexability issue (should be indexable but isn't)
    2. Page has bad HTTP status (404, 500, 302, etc.)
    3. Page is 301 redirect with 10+ inlinks (architectural issue)
    """
    # Indexability issue = cascade fail
    if indexability_issue:
        return True

    # Check HTTP status
    try:
        code = int(status_code)
        inlink_count = int(inlinks)

        # 404, 500, 503, etc. = broken page
        if code >= 400:
            return True

        # 302, 307 (temporary redirects) = problematic
        if code in [302, 307]:
            return True

        # 301 with high inlinks = architectural problem
        if code == 301 and inlink_count >= 10:
            return True

    except:
        return True

    return False

def title_exists_eval(title, indexability):
    """Evaluates title tag presence and length."""
    if indexability == "Indexable":
        title_str = str(title)
        if title_str == 'nan' or len(title_str) < 5 or len(title_str) > 69:
            return 1
    return 0

def mdesc_exists_eval(meta_description, indexability):
    """Evaluates meta description presence and length."""
    if indexability == "Indexable":
        desc_str = str(meta_description)
        if desc_str == 'nan' or len(desc_str) < 15 or len(desc_str) > 195:
            return 1
    return 0

def keyword_relevance_eval(text, keywords, indexability):
    """Evaluates if text contains keywords."""
    if indexability == "Indexable":
        text_lower = str(text).lower()
        if text_lower == 'nan':
            return 1
        keyword_list = [k.strip().lower() for k in str(keywords).split(',')]
        for keyword in keyword_list:
            if keyword in text_lower:
                return 0
        return 1
    return 0

def canonical_eval(status_code, canonical, page_url, indexability):
    """
    Evaluates canonical tag validity with smart near-match detection:
    1. Missing canonical = issue
    2. Near-duplicate URL (0.85+ similarity) = likely mistake
    3. Different enough URL = intentional (OK)
    """
    if status_code == 200 and indexability == "Indexable":
        # Missing canonical
        if pd.isna(canonical) or str(canonical).strip() == "" or str(canonical) == 'nan':
            return 1

        canonical_str = str(canonical).strip()
        page_url_str = str(page_url).strip()

        # Normalize for comparison
        canonical_norm = canonical_str.rstrip('/').lower()
        page_url_norm = page_url_str.rstrip('/').lower()

        # Perfect match (GOOD)
        if canonical_norm == page_url_norm:
            return 0

        # Calculate similarity
        similarity = SequenceMatcher(None, canonical_norm, page_url_norm).ratio()

        # Flag if too similar (0.85+) = likely mistake
        # Catches: trailing slashes, query params, case differences
        if similarity >= 0.85:
            return 1

        # Different enough = intentional cross-page canonical (OK)
        return 0

    return 0

def internal_linking_eval(inlinks, total_pages, indexability):
    """Evaluates internal link counts."""
    if indexability == "Indexable":
        try:
            inlinks_num = int(inlinks)
        except:
            inlinks_num = 0

        if total_pages < 21:
            if inlinks_num < (total_pages * 0.70):
                return 1
        elif total_pages < 100:
            if inlinks_num < 15:
                return 1
        elif total_pages >= 100:
            if inlinks_num < 51:
                return 1
    elif indexability == "Non-Indexable":
        try:
            inlinks_num = int(inlinks)
        except:
            inlinks_num = 0
        if inlinks_num > (total_pages * 0.10):
            return 1
    return 0

def content_length_eval(word_count, indexability):
    """Evaluates content word count."""
    if indexability == "Indexable":
        try:
            wc = int(word_count)
            if wc < 750:
                return 1
        except:
            return 1
    return 0

def page_speed_eval(lcp, cls, inp, indexability):
    """Evaluates Core Web Vitals."""
    if indexability == "Indexable":
        issues = 0
        try:
            if not pd.isna(lcp) and float(lcp) > 2500:
                issues += 1
        except:
            pass
        try:
            if not pd.isna(cls) and float(cls) > 0.1:
                issues += 1
        except:
            pass
        try:
            if not pd.isna(inp) and float(inp) > 200:
                issues += 1
        except:
            pass
        if issues > 0:
            return 1
    return 0

def url_optimization_eval(url, indexability):
    """Evaluates URL structure."""
    if indexability == "Indexable":
        issues = 0
        url_str = str(url)
        if not url_str.startswith("https"):
            issues += 1
        if "_" in url_str:
            issues += 1
        if any(char.isupper() for char in url_str):
            issues += 1
        if url_str.count('&') > 2:
            issues += 1
        if len(url_str) > 200:
            issues += 1
        if "session" in url_str.lower():
            issues += 1
        if issues > 0:
            return 1
    return 0

def heading_eval(h1, indexability):
    """Evaluates H1 presence."""
    if indexability == "Indexable":
        h1_str = str(h1)
        if h1_str == 'nan' or len(h1_str) < 5:
            return 1
    return 0

def h2_eval(h2, indexability):
    """Evaluates H2 presence."""
    if indexability == "Indexable":
        h2_str = str(h2)
        if h2_str == 'nan' or len(h2_str) < 1:
            return 1
    return 0

def duplication_eval(item, all_items, indexability):
    """Evaluates content duplication."""
    if indexability == "Indexable":
        item_str = str(item)
        if item_str == 'nan' or item_str == '':
            return 0
        count = sum(1 for other_item in all_items if str(other_item) == item_str)
        if count > 1:
            return 1
    return 0

def sitemap_eval(in_sitemap, indexability):
    """Evaluates sitemap inclusion."""
    if str(in_sitemap).lower() == 'yes' and indexability == "Non-Indexable":
        return 1
    if str(in_sitemap).lower() == 'no' and indexability == "Indexable":
        return 1
    return 0

# ====================================================================
# STEP 1: EVALUATE CRITICAL ISSUES (INDEXABILITY & HTTP STATUS)
# ====================================================================

print("\n📋 Step 1: Evaluating Critical Issues...")

total_pages = len(sf_dataframe)

# 1. Page Indexability (smart test)
print("   • Page Indexability (sitemap + inlinks)...")
sf_dataframe["Page Indexability"] = sf_dataframe.apply(
    lambda x: page_indexability_eval(
        x["Indexability"],
        x["Status Code"],
        x.get("In Sitemap", "no"),
        x["Inlinks"]
    ),
    axis=1
)

# 2. HTTP Response Status (enhanced with 301 inlink check)
print("   • HTTP Response Status (with 301 inlink threshold)...")
sf_dataframe["HTTP Responses"] = sf_dataframe.apply(
    lambda x: http_response_eval(x["Status Code"], x["Inlinks"]),
    axis=1
)

# 3. Determine cascade failure
print("   • Determining cascade failures...")
sf_dataframe["Should Cascade Fail"] = sf_dataframe.apply(
    lambda x: should_cascade_fail(
        x["Page Indexability"] == 1,
        x["HTTP Responses"] == 1,
        x["Status Code"],
        x["Inlinks"]
    ),
    axis=1
)

cascade_count = sf_dataframe["Should Cascade Fail"].sum()
print(f"\n✅ Critical issues evaluated")
print(f"   Pages with cascade failure: {cascade_count:,}")

# ====================================================================
# STEP 2: EVALUATE REMAINING SEO FACTORS (WITH CASCADE)
# ====================================================================

print("\n📋 Step 2: Evaluating Remaining SEO Factors...")

# 3. Title Existence
print("   • Title Tags...")
sf_dataframe["Does Title Exist"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else title_exists_eval(x["Title"], x["Indexability"]),
    axis=1
)

# 4. Meta Description Existence
print("   • Meta Descriptions...")
sf_dataframe["Does MDesc Exist"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else mdesc_exists_eval(x["Meta Description"], x["Indexability"]),
    axis=1
)

# 5-8. Keyword Relevance
print("   • Keyword Relevance...")
sf_dataframe["Title Brand Relevance"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else keyword_relevance_eval(x["Title"], BrandKeywords, x["Indexability"]),
    axis=1
)

sf_dataframe["Title Unbranded Relevance"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else keyword_relevance_eval(x["Title"], UnbrandedKeywords, x["Indexability"]),
    axis=1
)

sf_dataframe["MDesc Brand Relevance"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else keyword_relevance_eval(x["Meta Description"], BrandKeywords, x["Indexability"]),
    axis=1
)

sf_dataframe["MDesc Unbranded Relevance"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else keyword_relevance_eval(x["Meta Description"], UnbrandedKeywords, x["Indexability"]),
    axis=1
)

# 9. Canonicals (smart similarity detection)
print("   • Canonical Tags (with similarity detection)...")
sf_dataframe["Canonicals"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else canonical_eval(x["Status Code"], x["Canonical"], x["URL"], x["Indexability"]),
    axis=1
)

# 10. Sitemap
print("   • Sitemap...")
if "In Sitemap" in sf_dataframe.columns:
    sf_dataframe["Are There Sitemap Errors?"] = sf_dataframe.apply(
        lambda x: 1 if x["Should Cascade Fail"] else sitemap_eval(x["In Sitemap"], x["Indexability"]),
        axis=1
    )
else:
    sf_dataframe["Are There Sitemap Errors?"] = sf_dataframe.apply(
        lambda x: 1 if x["Should Cascade Fail"] else 0,
        axis=1
    )

# 11. Internal Linking
print("   • Internal Linking...")
sf_dataframe["Is Internal Linking Optimized"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else internal_linking_eval(x["Inlinks"], total_pages, x["Indexability"]),
    axis=1
)

# 12. Content Length
print("   • Content Length...")
sf_dataframe["Sufficient Body Content?"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else content_length_eval(x["Word Count"], x["Indexability"]),
    axis=1
)

# 13. Page Speed
print("   • Page Speed...")
sf_dataframe["Is Experience Optimized?"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else page_speed_eval(x["LCP"], x["CLS"], x["INP"], x["Indexability"]),
    axis=1
)

# 14. URL Optimization
print("   • URL Structure...")
sf_dataframe["Are URLs Optimized"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else url_optimization_eval(x["URL"], x["Indexability"]),
    axis=1
)

# 15. H1 Optimization
print("   • H1 Tags...")
sf_dataframe["Are Headings Optimized"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else heading_eval(x["H1"], x["Indexability"]),
    axis=1
)

# 16. H2 Optimization
print("   • H2 Tags...")
sf_dataframe["Are H2s Optimized"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else h2_eval(x["H2"], x["Indexability"]),
    axis=1
)

# 17-21. Duplication
print("   • Checking for duplicates...")

indexable_df = sf_dataframe[sf_dataframe["Indexability"] == "Indexable"]

sf_dataframe["Title Duplicates"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else duplication_eval(x["Title"], indexable_df["Title"].tolist(), x["Indexability"]),
    axis=1
)

sf_dataframe["Duplicate MDescs"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else duplication_eval(x["Meta Description"], indexable_df["Meta Description"].tolist(), x["Indexability"]),
    axis=1
)

sf_dataframe["H1 Duplicates"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else duplication_eval(x["H1"], indexable_df["H1"].tolist(), x["Indexability"]),
    axis=1
)

sf_dataframe["H2-1 Duplicates"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else duplication_eval(x["H2"], indexable_df["H2"].tolist(), x["Indexability"]),
    axis=1
)

# Placeholder for body content duplicates
sf_dataframe["Duplicate Paragraphs"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else 0,
    axis=1
)

print("✅ Traditional SEO evaluation complete!")

# ====================================================================
# STEP 3: CALCULATE TRADITIONAL SEO SCORE (PAGES THAT SHOULD BE INDEXABLE)
# ====================================================================

print("\n📊 Step 3: Calculating Traditional SEO Score...")

trad_seo_columns = [
    'Page Indexability', 'HTTP Responses', 'Does Title Exist', 'Does MDesc Exist',
    'Title Brand Relevance', 'Title Unbranded Relevance', 'MDesc Brand Relevance',
    'MDesc Unbranded Relevance', 'Canonicals', 'Are There Sitemap Errors?',
    'Is Internal Linking Optimized', 'Sufficient Body Content?', 'Is Experience Optimized?',
    'Are URLs Optimized', 'Are Headings Optimized', 'Are H2s Optimized',
    'Title Duplicates', 'Duplicate MDescs', 'H1 Duplicates', 'H2-1 Duplicates',
    'Duplicate Paragraphs'
]

# Calculate total issues per page
sf_dataframe['Traditional_SEO_Issues'] = sf_dataframe[trad_seo_columns].sum(axis=1)

# Calculate score per page (21 total factors)
sf_dataframe['Traditional SEO Score'] = 1 - (sf_dataframe['Traditional_SEO_Issues'] / 21)

# CORRECTED SCORING: Include all pages that SHOULD be indexable
# (Currently indexable OR failed indexability test)
indexable_pages = sf_dataframe[sf_dataframe['Indexability'] == 'Indexable']
should_be_indexable = sf_dataframe[
    (sf_dataframe['Indexability'] == 'Indexable') |
    (sf_dataframe['Page Indexability'] == 1)
]

if len(should_be_indexable) > 0:
    total_issues = should_be_indexable['Traditional_SEO_Issues'].sum()
    total_possible = len(should_be_indexable) * 21
    avg_trad_score = 1 - (total_issues / total_possible)
    issues_per_page = total_issues / len(should_be_indexable)

    print(f"✅ Traditional SEO Score: {avg_trad_score:.1%}")
    print(f"   Based on {len(should_be_indexable):,} pages that should be indexable:")
    print(f"     • Currently indexable: {len(indexable_pages):,}")
    print(f"     • Should be indexable (8+ inlinks or in sitemap): {len(should_be_indexable) - len(indexable_pages):,}")
    print(f"   Total issues: {int(total_issues):,}")
    print(f"   Issues per page: {issues_per_page:.1f}")
else:
    avg_trad_score = 0
    print("⚠️  No pages that should be indexable - cannot calculate score")

# ====================================================================
# STEP 4: AI SEARCH SCORING WITH CASCADE FAILURE
# ====================================================================

print("\n📋 Step 4: Evaluating AI Search Factors (WITH CASCADE FAILURE)...")

# AI factors - NOW WITH CASCADE FAILURE
# If page should be indexable but isn't, it can't be an AI source = all AI factors fail

print("   • Question Answering...")
sf_dataframe["Does Content Answer Questions"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else (
        0 if (x["Indexability"] == "Indexable" and ('?' in str(x.get('Title', '')) or x.get('Word Count', 0) > 1000)) else
        (1 if x["Indexability"] == "Indexable" else 0)
    ),
    axis=1
)

print("   • Authority...")
sf_dataframe["Is Content Authoritative"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else (
        0 if (x["Indexability"] == "Indexable" and x.get('Word Count', 0) > 1500) else
        (1 if x["Indexability"] == "Indexable" else 0)
    ),
    axis=1
)

print("   • Citations...")
sf_dataframe["Does Content Cite Sources"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else 0,
    axis=1
)

print("   • Structured Data...")
sf_dataframe["Is Structured Data Present"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else 0,
    axis=1
)

print("   • Topic Organization...")
sf_dataframe["Are Topics Organized"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else (
        0 if (x["Indexability"] == "Indexable" and len(str(x.get('H2', ''))) > 0 and str(x.get('H2', '')) != 'nan') else
        (1 if x["Indexability"] == "Indexable" else 0)
    ),
    axis=1
)

print("   • Conversational Tone...")
sf_dataframe["Is Content Conversational"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else 0,
    axis=1
)

print("   • Comprehensiveness...")
sf_dataframe["Is Content Comprehensive"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else (
        0 if (x["Indexability"] == "Indexable" and x.get('Word Count', 0) > 1000) else
        (1 if x["Indexability"] == "Indexable" else 0)
    ),
    axis=1
)

print("   • List Usage...")
sf_dataframe["Does Content Use Lists"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else 0,
    axis=1
)

print("   • Freshness...")
sf_dataframe["Is Content Fresh"] = sf_dataframe.apply(
    lambda x: 1 if x["Should Cascade Fail"] else 0,
    axis=1
)

print("✅ AI Search evaluation complete (with cascade failure)!")

# ====================================================================
# STEP 5: CALCULATE AI SEARCH SCORE (PAGES THAT SHOULD BE INDEXABLE)
# ====================================================================

print("\n📊 Step 5: Calculating AI Search Score...")

ai_search_columns = [
    'Does Content Answer Questions', 'Is Content Authoritative', 'Does Content Cite Sources',
    'Is Structured Data Present', 'Are Topics Organized', 'Is Content Conversational',
    'Is Content Comprehensive', 'Does Content Use Lists', 'Is Content Fresh'
]

sf_dataframe['AI_Search_Issues'] = sf_dataframe[ai_search_columns].sum(axis=1)
sf_dataframe['AI Search Score'] = 1 - (sf_dataframe['AI_Search_Issues'] / 9)

if len(should_be_indexable) > 0:
    # Recalculate should_be_indexable with new AI columns
    should_be_indexable = sf_dataframe[
        (sf_dataframe['Indexability'] == 'Indexable') |
        (sf_dataframe['Page Indexability'] == 1)
    ]

    total_ai_issues = should_be_indexable['AI_Search_Issues'].sum()
    total_ai_possible = len(should_be_indexable) * 9
    avg_ai_score = 1 - (total_ai_issues / total_ai_possible)

    print(f"✅ AI Search Score: {avg_ai_score:.1%}")
    print(f"   Based on {len(should_be_indexable):,} pages that should be indexable")
    print(f"   Total AI issues: {int(total_ai_issues):,}")
    print(f"   Issues per page: {total_ai_issues / len(should_be_indexable):.1f}")
else:
    avg_ai_score = 0
    print("⚠️  No pages that should be indexable - cannot calculate score")

# ====================================================================
# SUMMARY & DIAGNOSTICS
# ====================================================================

print("\n" + "="*70)
print("📈 ANALYSIS COMPLETE!")
print("="*70)

# Overall stats
non_indexable = sf_dataframe[sf_dataframe['Indexability'] == 'Non-Indexable']
print(f"\n📊 Site Overview:")
print(f"  Total Pages: {len(sf_dataframe):,}")
print(f"  • Currently Indexable: {len(indexable_pages):,} ({len(indexable_pages)/len(sf_dataframe)*100:.1f}%)")
print(f"  • Should Be Indexable: {len(should_be_indexable):,} ({len(should_be_indexable)/len(sf_dataframe)*100:.1f}%)")
print(f"  • Non-Indexable (Intentional): {len(non_indexable) - (len(should_be_indexable) - len(indexable_pages)):,}")

# Scores (pages that should be indexable)
print(f"\n🎯 SEO Scores:")
print(f"  Traditional SEO: {avg_trad_score:.1%}")
print(f"  AI Search: {avg_ai_score:.1%}")
print(f"\n  (Based on {len(should_be_indexable):,} pages that should be indexable)")

# Critical issues breakdown
print(f"\n⚠️  Critical Issues (Cascade Failures):")
indexability_fails = sf_dataframe[sf_dataframe['Page Indexability'] == 1]
http_fails = sf_dataframe[sf_dataframe['HTTP Responses'] == 1]
cascade_fails = sf_dataframe[sf_dataframe['Should Cascade Fail'] == True]

print(f"  Pages failing indexability test: {len(indexability_fails):,}")
if len(indexability_fails) > 0:
    in_sitemap = indexability_fails[indexability_fails['In Sitemap'] == 'yes']
    high_inlinks = indexability_fails[indexability_fails['Inlinks'] >= 8]
    print(f"    • In sitemap but non-indexable: {len(in_sitemap):,}")
    print(f"    • High inlinks (8+) but non-indexable: {len(high_inlinks):,}")

print(f"  Pages failing HTTP status test: {len(http_fails):,}")
if len(http_fails) > 0:
    redirects_301 = http_fails[http_fails['Status Code'] == 301]
    errors_4xx = http_fails[(http_fails['Status Code'] >= 400) & (http_fails['Status Code'] < 500)]
    errors_5xx = http_fails[http_fails['Status Code'] >= 500]
    print(f"    • 301 redirects with 10+ inlinks: {len(redirects_301):,}")
    print(f"    • 4xx errors: {len(errors_4xx):,}")
    print(f"    • 5xx errors: {len(errors_5xx):,}")

print(f"  Total cascade failures: {len(cascade_fails):,}")
print(f"    (These pages score 0% on BOTH Traditional SEO and AI Search)")

# Site architecture analysis
if len(should_be_indexable) > len(indexable_pages):
    difference = len(should_be_indexable) - len(indexable_pages)
    print(f"\n🚨 MAJOR ISSUE DETECTED:")
    print(f"  {difference:,} important pages are non-indexable but should be!")
    print(f"  These pages have 8+ inlinks or are in your sitemap")
    print(f"  This is causing your low SEO AND AI Search scores")

# JavaScript context
if 'JS_Share' in sf_dataframe.columns and 'Text Ratio' in sf_dataframe.columns:
    print(f"\n💡 JavaScript Analysis:")
    print(f"  Avg Text Ratio: {sf_dataframe['Text Ratio'].mean():.1f}%")
    print(f"  Avg JS Share: {sf_dataframe['JS_Share'].mean():.1f}%")
    js_heavy = sf_dataframe[(sf_dataframe['Text Ratio'] < 10) & (sf_dataframe['JS_Share'] > 30)]
    print(f"  JS-heavy pages: {len(js_heavy):,} ({len(js_heavy)/len(sf_dataframe)*100:.1f}%)")
    print(f"\n  Note: Site is JavaScript-heavy. Scores reflect JS-rendered content.")
    print(f"        Modern search engines (Google, ChatGPT) can render JS.")

print("\n✅ Data ready for export (run Cell 13+14)")
print("="*70)

# Store as crawlFrame for compatibility with export cells
crawlFrame = sf_dataframe.copy()
print("\n✅ Also created 'crawlFrame' variable")

In [ ]:
#@title Cell 13 - Export Analysis Results

#@title Cell 13+14 - Export Results (Combined)

import os

print("="*70)
print("📦 PREPARING ALL ANALYSIS FILES FOR DOWNLOAD")
print("="*70)

# ====================================================================
# STEP 0: CREATE OUTPUT DIRECTORY
# ====================================================================

output_dir = '/mnt/user-data/outputs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"\n✅ Created output directory: {output_dir}")

# ====================================================================
# STEP 1: VERIFY DATA
# ====================================================================

# Check which dataframe to use
if 'crawlFrame' in dir():
    df = crawlFrame.copy()
    print(f"\n✅ Using crawlFrame data")
elif 'sf_dataframe' in dir():
    df = sf_dataframe.copy()
    print(f"\n✅ Using sf_dataframe data")
else:
    print("\n❌ ERROR: No analysis data found")
    print("   Please run Cell 12 first")
    raise Exception("Analysis data not found")

print(f"   Total pages analyzed: {len(df):,}")

# Auto-detect URL column
url_col = None
for possible_name in ['Address', 'URL', 'address', 'url', 'Page URL', 'Link']:
    if possible_name in df.columns:
        url_col = possible_name
        break

if not url_col:
    print("❌ ERROR: Could not find URL column")
    raise Exception("URL column not found")

print(f"   URL column detected: '{url_col}'")

# Verify required scoring columns exist
required_cols = ['Traditional SEO Score', 'AI Search Score', 'Should Cascade Fail']
missing = [col for col in required_cols if col not in df.columns]

if missing:
    print(f"❌ ERROR: Missing columns: {missing}")
    print("   Please run Cell 12 first to generate scores")
    raise Exception("Required columns missing")

print(f"✅ All required columns present")

# ====================================================================
# STEP 2: CREATE DETAILED CSV EXPORT
# ====================================================================

print("\n" + "-"*70)
print("📊 PART 1: Creating detailed CSV export")
print("-"*70)

# Select columns for export (prioritize most important)
export_columns = [url_col]

# Add scoring columns
score_cols = ['Traditional SEO Score', 'AI Search Score', 'Should Cascade Fail',
              'Traditional_SEO_Issues', 'AI_Search_Issues']
export_columns.extend([c for c in score_cols if c in df.columns])

# Add core SEO columns
core_cols = ['Status Code', 'Indexability', 'Title', 'Meta Description',
             'H1', 'H2', 'Word Count', 'Inlinks', 'Canonical']
export_columns.extend([c for c in core_cols if c in df.columns])

# Add all evaluation columns (issue flags)
eval_cols = [c for c in df.columns if c in [
    'Page Indexability', 'HTTP Responses', 'Does Title Exist', 'Does MDesc Exist',
    'Title Brand Relevance', 'Title Unbranded Relevance', 'MDesc Brand Relevance',
    'MDesc Unbranded Relevance', 'Canonicals', 'Are There Sitemap Errors?',
    'Is Internal Linking Optimized', 'Sufficient Body Content?', 'Is Experience Optimized?',
    'Are URLs Optimized', 'Are Headings Optimized', 'Are H2s Optimized',
    'Title Duplicates', 'Duplicate MDescs', 'H1 Duplicates', 'H2-1 Duplicates',
    'Does Content Answer Questions', 'Is Content Authoritative', 'Does Content Cite Sources',
    'Is Structured Data Present', 'Are Topics Organized', 'Is Content Conversational',
    'Is Content Comprehensive', 'Does Content Use Lists', 'Is Content Fresh'
]]
export_columns.extend(eval_cols)

# Add performance columns if available
perf_cols = ['LCP', 'CLS', 'INP']
export_columns.extend([c for c in perf_cols if c in df.columns])

# Add JS analysis if available
js_cols = ['Text Ratio', 'JS_Share', 'JavaScript Size (Bytes)', 'HTML Size (Bytes)']
export_columns.extend([c for c in js_cols if c in df.columns])

# Remove duplicates while preserving order
seen = set()
export_columns = [x for x in export_columns if not (x in seen or seen.add(x))]

# Create export dataframe
export_df = df[export_columns].copy()

# Save CSV
csv_filename = os.path.join(output_dir, 'detailed_analysis.csv')
export_df.to_csv(csv_filename, index=False)

print(f"✅ CSV created: {len(export_df):,} pages")
print(f"   Columns: {len(export_columns)}")
print(f"   Location: {csv_filename}")

# ====================================================================
# STEP 3: CREATE MARKDOWN SUMMARY
# ====================================================================

print("\n" + "-"*70)
print("📄 PART 2: Creating markdown summary")
print("-"*70)

# Calculate summary statistics
total_pages = len(df)
indexable = df[df['Indexability'] == 'Indexable']
should_be_indexable = df[
    (df['Indexability'] == 'Indexable') |
    (df['Page Indexability'] == 1)
]
cascade_failures = df[df['Should Cascade Fail'] == True]

# Calculate scores
if len(should_be_indexable) > 0:
    avg_trad_score = should_be_indexable['Traditional SEO Score'].mean()
    avg_ai_score = should_be_indexable['AI Search Score'].mean()
else:
    avg_trad_score = 0
    avg_ai_score = 0

# Count specific issues
indexability_issues = len(df[df['Page Indexability'] == 1])
http_issues = len(df[df['HTTP Responses'] == 1])
errors_4xx = len(df[(df['Status Code'] >= 400) & (df['Status Code'] < 500)])

# Issue breakdown
issue_counts = {}
all_issue_cols = [
    'Page Indexability', 'HTTP Responses', 'Does Title Exist', 'Does MDesc Exist',
    'Title Brand Relevance', 'Title Unbranded Relevance', 'Canonicals',
    'Are There Sitemap Errors?', 'Is Internal Linking Optimized', 'Sufficient Body Content?',
    'Is Experience Optimized?', 'Are URLs Optimized', 'Are Headings Optimized',
    'Are H2s Optimized', 'Title Duplicates', 'Duplicate MDescs', 'H1 Duplicates'
]

for col in all_issue_cols:
    if col in df.columns:
        issue_counts[col] = df[col].sum()

# Sort by count
top_issues = sorted(issue_counts.items(), key=lambda x: x[1], reverse=True)[:10]

# Grade function
def get_grade(score):
    if score >= 0.9: return "A+ (Excellent)"
    elif score >= 0.8: return "A (Very Good)"
    elif score >= 0.7: return "B (Good)"
    elif score >= 0.6: return "C (Needs Improvement)"
    elif score >= 0.5: return "D (Poor)"
    else: return "F (Critical)"

# Create markdown report
markdown_content = f"""# SEO + AI Search Audit Summary

**Site:** {AnalyzedDomain if 'AnalyzedDomain' in dir() else 'Unknown'}
**Analysis Date:** {datetime.now().strftime('%B %d, %Y')}
**Total Pages Analyzed:** {total_pages:,}

---

## 📊 Overall Site Health

| Metric | Value |
|--------|-------|
| **Total Pages** | {total_pages:,} |
| **Currently Indexable** | {len(indexable):,} ({len(indexable)/total_pages*100:.1f}%) |
| **Should Be Indexable** | {len(should_be_indexable):,} ({len(should_be_indexable)/total_pages*100:.1f}%) |
| **Pages with Critical Issues** | {len(cascade_failures):,} ({len(cascade_failures)/total_pages*100:.1f}%) |

---

## 🎯 SEO Scores

### Traditional SEO Score: **{avg_trad_score:.1%}**
**Grade:** {get_grade(avg_trad_score)}

Based on {len(should_be_indexable):,} pages that should be indexable.

### AI Search Score: **{avg_ai_score:.1%}**
**Grade:** {get_grade(avg_ai_score)}

Measures readiness for generative AI platforms (ChatGPT, Perplexity, etc.)

---

## 🚨 Critical Issues

### Pages That Can't Be Indexed: **{indexability_issues:,}**
- These pages have 8+ inlinks OR are in your sitemap
- They SHOULD be indexable but aren't
- This prevents them from ranking in search AND being cited by AI

### HTTP Status Issues: **{http_issues:,}**
- Includes 404 errors: {errors_4xx:,}
- Problematic redirects
- Server errors

### Total Cascade Failures: **{len(cascade_failures):,}**
Pages that fail ALL tests (both Traditional SEO and AI Search)

---

## 📋 Top 10 Issues by Frequency

| Issue | Pages Affected | % of Site |
|-------|----------------|-----------|
"""

for issue_name, count in top_issues:
    pct = (count / total_pages) * 100
    markdown_content += f"| {issue_name} | {int(count):,} | {pct:.1f}% |\n"

# Add JavaScript analysis if available
if 'Text Ratio' in df.columns and 'JS_Share' in df.columns:
    avg_text_ratio = df['Text Ratio'].mean()
    avg_js_share = df['JS_Share'].mean()
    js_heavy = df[(df['Text Ratio'] < 10) & (df['JS_Share'] > 30)]

    markdown_content += f"""

---

## 💡 JavaScript Analysis

Your site is **JavaScript-heavy**:

| Metric | Value |
|--------|-------|
| **Avg Text Ratio** | {avg_text_ratio:.1f}% |
| **Avg JS Share** | {avg_js_share:.1f}% |
| **JS-Heavy Pages** | {len(js_heavy):,} ({len(js_heavy)/total_pages*100:.1f}%) |

**What this means:**
- Most content is rendered by JavaScript (not in HTML source)
- Modern search engines (Google, ChatGPT) can render JavaScript ✅
- Legacy crawlers may miss content ❌
- These scores reflect JS-rendered content (Screaming Frog with JS enabled)

"""

# Add action plan
markdown_content += f"""

---

## 🎯 Recommended Action Plan

### Phase 1: Critical (Do Immediately)
1. **Fix {indexability_issues:,} non-indexable pages**
   - Review pages with 8+ inlinks that are blocked
   - Check canonical tags - are they pointing to wrong URLs?
   - Verify robots.txt and meta robots tags

2. **Fix {errors_4xx:,} broken pages (404 errors)**
   - Create 301 redirects to relevant pages
   - Update internal links
   - Remove from navigation if obsolete

### Phase 2: High Priority
3. **Optimize the {len(indexable):,} indexable pages**
   - Fix title/description issues
   - Add missing H1/H2 tags
   - Improve content length (aim for 750+ words)

### Phase 3: Medium Priority
4. **Improve AI Search readiness**
   - Add structured data (Schema.org)
   - Improve content comprehensiveness
   - Add internal citations and sources

### Phase 4: Long-term Improvements
5. **Site architecture review**
   - Review canonical strategy
   - Optimize internal linking
   - Monitor Core Web Vitals

---

## 📁 Files Generated

1. **detailed_analysis.csv** - Complete page-by-page analysis with all scores and issues
2. **audit_summary.md** - This executive summary (you're reading it!)

---

## 🔍 Next Steps

1. **Review the detailed_analysis.csv** to see page-by-page issues
2. **Filter by "Should Cascade Fail = TRUE"** to see critical pages first
3. **Sort by "Inlinks"** to prioritize high-value pages
4. **Check canonical targets** to understand indexability issues

---

*Generated by SEO + AI Search Audit Tool*
*Powered by Screaming Frog data with JavaScript rendering enabled*
"""

# Save markdown
markdown_filename = os.path.join(output_dir, 'audit_summary.md')
with open(markdown_filename, 'w', encoding='utf-8') as f:
    f.write(markdown_content)

print(f"✅ Markdown created")
print(f"   Location: {markdown_filename}")

# ====================================================================
# STEP 4: PRESENT FILES TO USER
# ====================================================================

print("\n" + "="*70)
print("✅ ALL FILES READY FOR DOWNLOAD!")
print("="*70)

from google.colab import files as colab_files

print("\nDownloading files...")

# Download CSV
colab_files.download(csv_filename)
print(f"✅ Downloaded: detailed_analysis.csv")

# Download Markdown
colab_files.download(markdown_filename)
print(f"✅ Downloaded: audit_summary.md")

print("\n" + "="*70)
print("🎉 EXPORT COMPLETE!")
print("="*70)
print(f"\nYou now have:")
print(f"  1. detailed_analysis.csv ({len(export_df):,} pages)")
print(f"  2. audit_summary.md (executive summary)")
print(f"\nOpen the CSV in Excel/Google Sheets to:")
print(f"  • Filter by 'Should Cascade Fail = TRUE' for critical issues")
print(f"  • Sort by 'Inlinks' to prioritize important pages")
print(f"  • Review 'Traditional SEO Score' and 'AI Search Score' by page")
print("="*70)